<a href="https://colab.research.google.com/github/vishal-suri/ExData_Plotting1/blob/master/Vishal_Suri_Full_Code_NLP_RAG_Project_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Submission by - Vishal Suri

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 113.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 224.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 136.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 221.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip uninstall numpy -y
!pip install numpy==1.26.4 -q #Removing the most advanced versin of numpy and adding 1.26.4 as that was compatible with the sentence transformer package used here
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 -q

Found existing installation: numpy 2.4.2
Uninstalling numpy-2.4.2:
  Successfully uninstalled numpy-2.4.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [ ]:
# uncomment and run the following lines for Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

# Question Answering using LLM

##Defining the Response Generator

#### Downloading and Loading the model

Let us use the Mistral 7 B model version 0.2 from Hugging Face (HF) as it has been proven to perform well in several tests, particularly on the 7B parameters leaderboard on HF.

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
#Download model from Hugging Face
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [ ]:
#uncomment the below snippet of code if the runtime is connected to GPU.
llm = Llama(
    model_path=model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [ ]:
#uncomment the below snippet of code if the runtime is connected to CPU only.
#llm = Llama(
#    model_path=model_path,
#    n_ctx=1024,
#    n_cores=-2
#)

##Response Function that helps generate response to a query

In [ ]:
#Define a function to generate response from llm's own sources
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k,
      stop=["</s>"],echo=False
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query="What is the protocol of managing Sepsis in a critical care unit?"

In [ ]:
print(response(query))

Llama.generate: prefix-match hit




Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and management in a critical care unit. The following are the general steps for managing sepsis in a critical care unit:

1. Early recognition and suspicion: Sepsis should be suspected in any patient who has an infection and is showing signs of organ dysfunction, such as altered mental status, respiratory distress, or decreased urine output.
2. Rapid assessment and resuscitation: Once sepsis is suspected, the patient should undergo a rapid assessment to determine their


##Observations:
1. The response does seem somewhat relevant to the query but seems incomplete.

2. The response stops mid-sentence indicating that max_tokens parameter is not adequate and that the LLM is not able to balance the response such that it adequately and completely summarizes the answer while mainintaing response length within the allowed token limit.

3. Need a domain expert to understand the correctness and groundedness of the response.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(query))

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:

1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea


##Observations:
1. The response goes into a description of what Appendix is and what Appendicitis is, and its symptoms.

2. It does answer whether medicine will work, and what kind of surgery is needed.

3.  It also still stops mid-sentence.  It could not balance the answer by reducing the words in answering the symptoms and using that space to respond about surgery versus medicine topic.   

4. Domain experts are still needed to judge whether the provided response is even partially accurate.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(query))

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.

The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.


##Observations:
1. The response again is too short and stops mid-sentence.  

2.  It delves into the causes and symptoms of patchy hair loss, but does not talk about treatments or solutions for the same.

3.  Increasing the max_tokens and prompting the llm to not stop mid-sentence could fix the issue.

4.  Domain expertise will be needed to determine the relevance and accuracy of the partial response.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

> Add blockquote



In [ ]:
query="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(query))


Llama.generate: prefix-match hit




A person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.
2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a


##Observations:
1. The Response generator once again generates a partial response that stops mid sentence.

2.  It briefly talks of the basic treatment needed, but does not get adequate length to describe the treatment in detail.

3.  The response does not differentiate between temporary versus permanent brain damage diagnosis or treatment.

4.  Domain experts will be needed to determine the completeness and accuracy of even the partial response.

5.  The default max_tokens is too low.  Need to increase it and ask the model to not stop mid-sentence for it to produce meaningful response.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(query))

Llama.generate: prefix-match hit




First and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.
2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.
3. Immobilize the leg: Use a splint, sl


##Observations:
1. The max_tokens default value seems too low to generate meaningful response to such questions.  Increasing the max_tokens and asking the prompt to not stop mid-sentence to get a relevant and meaningful response.

2. The response produced seems somewhat relevant to the question of fracture.

3. It provides the precautions needed partially, but is not able to talk of the treatments as it does not balance the summarization enough.

4. The response stops mid-sentence.

5. The response also seems less like how it will assist a doctor, and more like how it will instruct a layman on what to do.

6. A domain expert will be needed to determine veracity, thoroughness, and accuracy of the response.


# Question Answering using LLM with Prompt Engineering

####The LLM without prompt engineering generated partial answers, that stopped mid sentence.  Let us use the following strategies for improved prompt engineering : a) Improve the prompt to provide the right system context to the LLM; ask it to not hallucinate; and to not stop mid sentence; b) Increase token length to help it provide more complete answers; c) Investigate improving creativity of the answer by changing the temperature parameter.

#Step 1 : Let us first start with improving the prompt through addition of the system context

In [ ]:
qna_system_message = """
You are a medical assistant whose work is to assist a medical doctor and provide the appropriate answers to the question.
User questions will begin with the token: ###Question.
If you are not able to find the answer to the question, then do not make-up an answer.  Do not hallucinate.  Instead, respond "I don't know".
Do not stop mid-sentence. Please respond with complete sentences only.
"""

In [ ]:
user_message_template = """
###Question
{question}
"""

In [ ]:
#Define a function that incorporates the system message and user message
#Then utilizes the llm to generate responses
def new_response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message, user_message_template
    user_message = user_message_template.replace('{question}', query)
    prompt = qna_system_message + '\n' + user_message
    model_output = llm(
      prompt=prompt,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k,
      stop=["</s>"],echo=False
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query="What is the protocol of managing Sepsis in a critical care unit?"
print(new_response(query))

Llama.generate: prefix-match hit



I'd be happy to help answer your question regarding the protocol for managing sepsis in a critical care unit. The treatment for sepsis involves addressing its underlying causes while providing supportive care to maintain organ function and prevent complications. Here are some general steps that may be taken:

1. Early recognition and prompt initiation of antibiotic therapy: This is crucial as delaying antibiotic administration can worsen the condition and increase mortality. The choice of antibiotics depends on the suspected pathogen and local resistance patterns.

2. Fluid resuscitation: Sepsis can lead


##Observations:
1. The additional system prompt has simply made the LLM more polite in the response instead of adding completeness to the response.

2.  In trying to add generic polite language, it loses more words in that and produces even fewer meaningful sentences.

3.  Unfortunately, it still stops mid sentence even when prompted to not do so.

4.  The max_tokens default value seems too low to generate meaningful response to such questions.  Increasing the max_tokens could generate better response.



### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(new_response(query))

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The common symptoms for appendicitis include:
1. Abdominal pain, usually starting around the navel area and then shifting to the lower right side of the abdomen.
2. Loss of appetite.
3. Nausea and vomiting.
4. Fever.
5. Constipation or diarrhea.
6. Inability to pass gas or have a bowel movement.
7. Abdominal swelling.


#Observations:
1.  The additional prompt seems to have worked here at least in creating a more complete response that has only full sentences.
2.  However, the response only provides the symptoms of Appendicitis and not the treatments.
3.  It also does not address the medicine versus surgery question, or the type of surgery question.

4.  Essentially, it is not able to balance the response to summarize the symptoms in fewer words and utilize those words to address the remaining queries within the allowable token limit.

5.  Increasing the max_tokens should help with completeness of the response.

6.  Domain expert is still needed to determine the accuracy of the response.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(new_response(query))

Llama.generate: prefix-match hit



I'd be happy to help answer your question regarding sudden patchy hair loss, also known as alopecia areata. This condition is characterized by round or oval patches of total hair loss on the scalp, beard, or other areas of the body. The exact cause of alopecia areata is unknown, but it's believed to be an autoimmune disease where the immune system attacks the hair follicles.

There are several treatments that have been shown to be effective in addressing sudden patchy hair loss due to alopecia areata:

1. Corticoster


#Observations:
1.  Unfortunately, this response was again more like the response to query 1.

2.  In trying to add more polite language, the LLM lost even more space to address the original query completely.

3.  The response slightly touches on the cause and other names for the patchy hair loss disease.  However, the remedies and causes both stay incomplete.  The response also stops mid-sentence despite being asked to not do so.

4.  Improving max_tokens should be able to handle such issue better.

5.  Domain expert is still needed to verify accuracy of the response.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?


In [ ]:
query="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(new_response(query))


Llama.generate: prefix-match hit



Based on the information available to me, I would recommend several treatments for a person who has sustained a physical injury to brain tissue, leading to temporary or permanent impairment of brain function. These treatments may include:

1. Medications: Depending on the specific symptoms and conditions, various medications may be prescribed to manage seizures, control pain, reduce inflammation, improve cognitive function, and address other complications.

2. Rehabilitation therapy: A combination of physical, occupational, speech, and cognitive rehabilitation therapies can help individuals regain lost skills and functions, improve overall functioning


#Observations:
1. The LLM goes into slight more detail about the specific treatments, however the overall response still seems quite generic.

2.  However, it does not talk of what specific medications to be provided.

3.  It does not distinguish between termporary or permanent impairment treatment or rehabilitation.

4.  Need to increase the max_tokens to improve the completeness of the response.

5.  Domain expert is still needed to determine the accuracy of even the partial response.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(new_response(query))

Llama.generate: prefix-match hit



A fractured leg is a serious injury that requires prompt medical attention. Here are some necessary precautions and treatment steps for a person with a fractured leg during a hiking trip:

1. **Stabilize the injury**: Try to prevent any further movement of the injured leg by using a splint, sling, or other immobilizing device. This will help reduce pain and prevent worsening of the injury.
2. **Provide first aid**: Apply a clean pad to the wound and apply gentle pressure with a sterile dressing to control bleeding. Splint the leg above and below the


#Observations:
1.  The response is less complete than with the original response to this query.

2.  Addition of the detailed prompt has not helped much in improving the specificity of the response or completeness or even in prompting the response to not stop mid-sentence.

3.  Once again, increasing the max_tokens seems to be the best way to remedy the completeness of the response.

4.  A domain expert will still be needed to assess the accuracy of the response.

#Step 2 : Let us increase the maximum allowed tokens compared to default values that should improve the completeness of the response here.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query="What is the protocol of managing Sepsis in a critical care unit?"
print(new_response(query,max_tokens=256))


I'd be happy to help answer your question regarding the protocol for managing sepsis in a critical care unit. The treatment for sepsis involves addressing its underlying causes while providing supportive care to maintain organ function and prevent complications. Here are some general steps that may be taken:

1. Early recognition and prompt initiation of antibiotic therapy: This is crucial as delaying antibiotic administration can worsen the condition and increase mortality. The choice of antibiotics depends on the suspected pathogen and local resistance patterns.

2. Fluid resuscitation: Sepsis often leads to hypovolemia, so intravenous fluids are administered to maintain adequate blood pressure and organ perfusion. Crystalloids like normal saline or lactated Ringer's solution are commonly used for initial resuscitation.

3. Vasopressor support: In cases of septic shock, where blood pressure remains low despite fluid resuscitation, vasopressors may be required to maintain adequate pe

#Observations:
1.  The response certainly has better completeness with the increase in max_tokens.
2.  The response does seem more specific with points like antibiotics, fluid resucitation, vasopressors.
3.  Unforunately, it still is incomplete and it still stops mid-sentence, indicating that it needs more allowable limit for max_tokens.
4.  A domain expert is still require to judge adequacy and accuracy of the response compared to real life.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(new_response(query,max_tokens=256))

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The common symptoms for appendicitis include:
1. Sudden pain in the lower right abdomen, which may begin as mild and then gradually worsens over several hours.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Vomiting.
4. Fever, which may start after the abdominal pain begins.
5. Constipation or diarrhea.
6. Abdominal swelling and tenderness.
7. Inability to pass gas or have a bowel movement.
8. Pain in the lower back, on the right side.
9. Feeling restless or unable to find a comfortable position.

Appendicitis cannot be cured via medicine alone. If left untreated, an inflamed appendix can rupture and spill infectious materials into the abdominal cavity, leading to peritonitis, a serious inflammation of the abdominal lining. This condition requires immediate surgical intervention. The standard surgical procedure for treating appendic

#Observations:
1. The response is more complete with increase in allowable tokens.  It now lists 9 symptoms of Appendicitis instead of 7. It also adds a comment about medicine versus surgery.
2.  Unfortunately, the response about surgical procedure is still partial, and the response stops mid-sentence as it ran out of tokens.
3.  A domain expert is still needed to judge the accuracy and thoroughness of the response here.
4.  Increasing the max_tokens further should fix this issue with the response.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(new_response(query,max_tokens=256))

Llama.generate: prefix-match hit



I'd be happy to help answer your question. However, I must first clarify that I am a medical assistant and not a doctor. Therefore, I cannot provide definitive medical advice or treatments. I can, however, share some information about possible causes and potential solutions for sudden patchy hair loss.

Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes the immune system to attack hair follicles, leading to hair loss in small patches on the scalp or other areas of the body. The exact cause of alopecia areata is not known, but it is believed to be related to genetics and environmental factors.

There are several treatments that may help address sudden patchy hair loss:

1. Corticosteroids: These medications can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally.
2. Minoxidil: This medication is a vasodilator that can help increase blood flow to the hair follicl

#Observations:
1. The response is this time able to delve deeper into both the causes as well as potential remedies of sudden patchy hair loss.
2.  However, the response is still not complete, and needs more allowance in max_tokens to completely provide the answer.
3.  Curiously, while previous versions of query 3 response provided many more causes, this version lists only genetics and environmental factors as the causes of patchy localized bald spots.
4.  Thus, an expert is still needed to determine the accuracy and exhaustiveness of the response.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(new_response(query,max_tokens=256))

Llama.generate: prefix-match hit



Based on the information available to me, I would recommend several treatments for a person who has sustained a physical injury to brain tissue, leading to temporary or permanent impairment of brain function. These treatments may include:

1. Medications: Depending on the specific symptoms and conditions, various medications may be prescribed to manage seizures, control pain, reduce inflammation, improve cognitive function, and address other complications.

2. Rehabilitation therapy: A combination of physical, occupational, speech, and cognitive rehabilitation therapies can help individuals regain lost skills and functions, improve overall functioning, and enhance their quality of life.

3. Assistive devices: Depending on the extent and nature of the brain injury, assistive devices such as wheelchairs, communication aids, and mobility aids may be recommended to help individuals perform daily activities more effectively and safely.

4. Surgery: In some cases, surgical intervention may 

#Observations:
1. The response is certainly more complete and provides more treatments to the patients of temporary or permanent brain impairment.
2.  However, the treatments still sound generic and could potentially be more specific for medical doctors.
3.  The response still stops mid-sentence, and hard to tell whether it would have produced more bullet points.  Increase in max_tokens could help solve for that.
4.  A domain expert can further comment or certify the accuracy and exhaustiveness of this response.  It certainly seems relevant to the query asked.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(new_response(query,max_tokens=256))

Llama.generate: prefix-match hit



A fractured leg is a serious injury that requires prompt medical attention. Here are some necessary precautions and treatment steps for a person with a fractured leg during a hiking trip:

1. **Stabilize the injury**: Try to prevent any further movement of the injured leg by using a splint, sling, or other immobilizing device. This will help reduce pain and prevent worsening of the injury.
2. **Provide first aid**: Apply a clean pad to the wound and apply gentle pressure with a sterile dressing to control bleeding. Splint the leg above and below the injury if possible.
3. **Transport the person safely**: If the person is unable to walk, use a stretcher or other carrying device to transport them down the mountain or trail safely. Avoid jolting or bouncing the injured leg during transportation.
4. **Seek medical attention**: Once you have reached a medical facility, the healthcare professional will assess the severity of the fracture and determine the appropriate treatment. This may inc

#Observations:
1. Once again, the response is more complete.  However, it definitely sounds more generic as if addressed to lay-men as audience instead of medical doctors being the audience.
2.  It's hard to say if it would have produced more complete response with increase in tokens, but it might.  While the response talks of general treatment and precautions, it does not go into details of care, recovery, and medication needed for such patients.
3.  Decrease in temperature could potentiall produce more creative or niche responses than the response received so far.
4.  Increase in tokens could also help produce a more complete response.
5.  Domain expert need is still there to certify accuracy and completeness of the response.  It certainly seems relevant to the query asked.

#Step 3 : Let us increase the maximum allowed tokens further to 512  to improve the completeness of the response even more.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query="What is the protocol of managing Sepsis in a critical care unit?"
print(new_response(query,max_tokens=512))

Llama.generate: prefix-match hit



I'd be happy to help answer your question regarding the management of sepsis in a critical care unit. The protocol for managing sepsis in a critical care setting typically involves several key steps:

1. Early recognition and diagnosis: This includes recognizing the signs and symptoms of sepsis, such as fever, tachycardia, respiratory distress, and altered mental status. Laboratory tests, including blood cultures and lactate levels, can also help confirm the diagnosis.
2. Immediate fluid resuscitation: Fluid resuscitation is a critical component of sepsis management in the early stages. The goal is to maintain adequate tissue perfusion and prevent organ dysfunction. This may involve administering intravenous fluids quickly to restore intravascular volume.
3. Antibiotic therapy: Broad-spectrum antibiotics should be initiated as soon as possible, ideally within one hour of recognition and diagnosis of sepsis. The choice of antibiotics will depend on the suspected source of infection and

#Observations:
1.  This time the model lists a 9-step protocol instead of the earlier 3 steps, providing a near complete answer to the query.
2.  Need a domain expert to provide more specific responses, and assess the quality of response here.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(new_response(query,max_tokens=512))

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The common symptoms for appendicitis include:
1. Sudden pain in the lower right abdomen, which may begin as mild and then gradually worsens over several hours.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Vomiting.
4. Fever, which may start after the abdominal pain begins.
5. Constipation or diarrhea.
6. Abdominal swelling and tenderness.
7. Inability to pass gas or have a bowel movement.
8. Pain in the lower back, on the right side.
9. Feeling restless or unable to find a comfortable position.

Appendicitis cannot be cured via medicine alone. If left untreated, an inflamed appendix can rupture and spill infectious materials into the abdominal cavity, leading to peritonitis, a serious inflammation of the abdominal lining. This condition requires immediate surgical intervention. The standard surgical procedure for treating appendic

Observations:
1. This is one of the most complete answers so far in the session.
2.  The response provides both the exhaustive symptoms as well as opines on the medicine versus surgery treatment options.
3.  This response did not increase much in length from the previous response, so likely that it wont get impacted with any further increase in max_token parameter.
4.  The temperature parameter could be tested further to assess if it can provide more creative and niche responses.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(new_response(query,max_tokens=512))

Llama.generate: prefix-match hit



I'd be happy to help answer your question. However, I must first clarify that I am a medical assistant and not a doctor. Therefore, I cannot provide definitive medical advice or treatments. I can, however, share some information about possible causes and potential solutions for sudden patchy hair loss.

Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes the immune system to attack hair follicles, leading to hair loss in small patches on the scalp or other areas of the body. The exact cause of alopecia areata is not known, but it is believed to be related to genetics and environmental factors.

There are several treatments that may help address sudden patchy hair loss:

1. Corticosteroids: These medications can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally.
2. Minoxidil: This medication is a vasodilator that can help increase blood flow to the hair follicl

#Observations:
1. The response lists all main treatments, as well as the key cause of patchy bandness.
2.  While providing the response, it also added a few clarifications that the response LLM is not a doctor, and that the treatment can vary person to person.
3.  The response seems complete in itself, as well as relevant to the query asked.  A domain expert can further help judge the completeness and accuracy of the final response.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(new_response(query,max_tokens=512))

Llama.generate: prefix-match hit



Based on the information available to me, I would recommend several treatments for a person who has sustained a physical injury to brain tissue, leading to temporary or permanent impairment of brain function. These treatments may include:

1. Medications: Depending on the specific symptoms and conditions, various medications may be prescribed to manage seizures, control pain, reduce inflammation, improve cognitive function, and address other complications.

2. Rehabilitation therapy: A combination of physical, occupational, speech, and cognitive rehabilitation therapies can help individuals regain lost skills and functions, improve overall functioning, and enhance their quality of life.

3. Surgery: In some cases, surgical intervention may be necessary to remove hematomas or other lesions that are causing pressure on the brain, alleviate seizures, or address other complications.

4. Assistive devices: Depending on the extent and nature of the injury, assistive devices such as wheelcha

#Observations:
1. The response generates a 7 point set of treatments instead of the previous 5 types.
2.  The additional therapies both talk about support groups and psychological counseling.  Thus, psychological well-being related points have come in as additional to the overall treatment recommendations for those with brain impairment.
3. Domain expert can further help ratify to the accracy of the information submitted.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(new_response(query,max_tokens=512))

Llama.generate: prefix-match hit



A fractured leg is a serious injury that requires prompt medical attention. Here are some necessary precautions and treatment steps for a person with a fractured leg during a hiking trip:

1. **Stabilize the injury**: Try to prevent any further movement of the injured leg by using a splint, sling, or other immobilizing device. This will help reduce pain and prevent worsening of the injury.
2. **Provide first aid**: Apply a clean pad to the wound and apply gentle pressure with a sterile dressing to control bleeding. Splint the leg above and below the injury if possible.
3. **Transport the person safely**: If the person is unable to walk, use a stretcher or other carrying device to transport them down the mountain or trail safely. Avoid jolting or bouncing the injured leg during transportation.
4. **Seek medical attention**: Once you have reached a medical facility, the healthcare professional will assess the severity of the fracture and determine the appropriate treatment. This may inc

#Observations:
1.  The response has increased from 5 points earlier to 8 points for the treatment.  
2.  The treatment steps include followup care as well as commentary on healthy lifestyle.  Thus, the regular medication or treatment steps seem to have been all accounted for.  The response is therefore potentially complete.
3.  However, a domain expert can further comment about more detailed meaning and depth for the models.

#Step 4 : Let us increase max tokens further to improve the completeness of the response even more.

### Query 1: What is the protocol for managing sepsis in a critical care unit?



In [ ]:
query="What is the protocol of managing Sepsis in a critical care unit?"
print(new_response(query,max_tokens=1024))

Llama.generate: prefix-match hit



I'd be happy to help answer your question regarding the protocol for managing sepsis in a critical care unit. The treatment for sepsis involves addressing its underlying causes while providing supportive care to maintain organ function and prevent complications. Here are some general steps that may be taken:

1. Early recognition and prompt initiation of antibiotic therapy: This is crucial as delaying antibiotic administration can worsen the condition and increase mortality. The choice of antibiotics depends on the suspected pathogen and local resistance patterns.

2. Fluid resuscitation: Sepsis can lead to hypovolemia, so intravenous fluids are often administered to maintain adequate blood pressure and organ perfusion. Crystalloids like normal saline or lactated Ringer's solution are commonly used for initial resuscitation.

3. Vasopressor support: In cases of septic shock, where blood pressure remains low despite fluid resuscitation, vasopressors may be required to maintain adequate

#Observations:
1.  This certainly provides a complete response to the query.  The response is more comprehensive than any response to this query so far.  It also completes all sentences well.
2.  The response also looks relevant to the query asked.
3.  A validation through a domain expert, or a validation check through a different llm could help determine if the key critical steps for the query are covered or not.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(new_response(query,max_tokens=1024))

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The common symptoms for appendicitis include:
1. Sudden pain in the lower right abdomen, which may begin as mild and then gradually worsens over several hours.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Vomiting.
4. Fever, which may be low-grade at first but can rise to very high temperatures as the condition progresses.
5. Constipation or diarrhea.
6. Abdominal swelling and tenderness.
7. Pain in the lower right quadrant of the abdomen when moving or pressing on that area.
8. Inability to pass gas or have a bowel movement.
9. Feeling restless or unable to find a comfortable position due to pain.

Appendicitis cannot be cured via medicine alone, as the inflammation and swelling of the appendix can obstruct the blood flow, leading to tissue death and potential rupture. If left untreated, an appendix rupture can lead to peritoniti

#Observations:
1. The response not only provides the same 9 bullet points as earlier; it addresses the medicine versus surgery question.  More importantly, it goes on to describe appendectomy as the key surgery and the kinds of appendectomies executed.  Thus, it seems a more complete and exhaustive response.
2. A validation through a domain expert, or a validation check through a different llm could help determine if the key critical steps for the query are covered or not.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(new_response(query,max_tokens=1024))

Llama.generate: prefix-match hit



I'd be happy to help answer your question. However, I must first clarify that I am a medical assistant and not a doctor. Therefore, I cannot provide definitive medical advice or treatments. I can, however, share some information about possible causes and potential solutions for sudden patchy hair loss.

Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes the immune system to attack hair follicles, leading to hair loss in small patches on the scalp or other areas of the body. The exact cause of alopecia areata is not known, but it is believed to be related to genetics and environmental factors.

There are several treatments that may help address sudden patchy hair loss:

1. Corticosteroids: These medications can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally.
2. Minoxidil: This medication is a vasodilator that can help increase blood flow to the hair follicl

#Observations:
1. The response feels complete as it not only has all five bullet points; it completes the previous sentences left unfinished.
2. Interestingly, while the response seems complete, the answer for one of the bullet points has moved from immune system modulators to alternative therapies.  Thus, it is altering some therapies that are probably less documented.
3. A domain expert will be needed to determine the completeness and also the utility of each of the bullet points.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(new_response(query,max_tokens=1024))

Llama.generate: prefix-match hit



Based on the information available to me, I would recommend several treatments for a person who has sustained a physical injury to brain tissue, leading to temporary or permanent impairment of brain function. These treatments may include:

1. Medications: Depending on the specific symptoms and conditions, various medications may be prescribed to manage seizures, control pain, reduce inflammation, improve cognitive function, and address other complications.

2. Rehabilitation therapy: A combination of physical, occupational, speech, and cognitive rehabilitation therapies can help individuals regain lost skills and functions, improve overall functioning, and enhance their quality of life.

3. Surgery: In some cases, surgical intervention may be necessary to remove hematomas or other lesions that are causing pressure on the brain, alleviate seizures, or address other complications.

4. Assistive devices: Depending on the extent and nature of the injury, assistive devices such as wheelcha

#Observations:
1. The response provides 7 bullet points as earlier, and adds caution about consulting a doctor based on each individual patient's condition.
2.  Interestingly, it replaces the psychological support bullet point with a new alternative therapy bullet point.  Thus, there are some changes that it does to the original response.
3.  Need a domain expert to evaluate the accuracy of the final bullet point versus others. Noting the source link to each of these will help the expert in determining the accuracy of each bullet point.  Overall, the response does seem relevant to the query asked.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(new_response(query,max_tokens=1024))

Llama.generate: prefix-match hit



A fractured leg is a serious injury that requires prompt medical attention. Here are some necessary precautions and treatment steps for a person with a fractured leg during a hiking trip:

1. **Stabilize the injury**: Try to prevent any further movement of the injured leg by using a splint, sling, or other immobilizing device. This will help reduce pain and prevent worsening of the injury.
2. **Provide first aid**: Apply a clean pad to the wound and apply gentle pressure with a sterile dressing to control bleeding. Splint the leg above and below the injury if possible.
3. **Transport the person safely**: If the person is unable to walk, use a stretcher or other carrying device to transport them down the mountain or trail safely. Avoid jolting or bouncing the injured leg during transportation.
4. **Seek medical help**: Once you have reached a medical facility, the healthcare professional will assess the severity of the fracture and determine the appropriate treatment. This may include 

#Observations:
1.  In this case, the response is almost identical to the response with smaller max_token allowance.  It repeats the same 8 bullets with no further addition to the sumamry.
2.  For this particular query, the response window is no different with increase in max tokens.

#Step 5 : Let us increase the temperature parameter to get less deterministic and more creative, balanced or niche responses to the prompt.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query="What is the protocol of managing Sepsis in a critical care unit?"
print(new_response(query,max_tokens=1024,temperature=0.5))

Llama.generate: prefix-match hit



As a medical assistant, I would refer to the current sepsis guidelines established by reputable organizations such as the Surviving Sepsis Campaign (SSC) or the European Society of Intensive Care Medicine (ESICM). These guidelines provide evidence-based recommendations for managing sepsis in critical care units.

The protocol typically includes the following steps:

1. Early recognition and diagnosis: Identify sepsis early by recognizing signs and symptoms, such as fever or hypothermia, tachycardia, tachypnea, altered mental status, and lactic acidosis. Use validated scoring systems like the Sequential Organ Failure Assessment (SOFA) score to help diagnose sepsis.
2. Initial resuscitation: Administer oxygen, fluids, and vasopressors as needed to maintain adequate tissue perfusion and organ function. Target a mean arterial pressure (MAP) of at least 65 mmHg and a central venous oxygen saturation (ScvO2) of greater than 70%.
3. Source control: Identify and address the source of infectio

#Observations:
1. This time the response not only includes the specific protocols, it adds a few points to the protocol and provides the sources of its data as well.
2. It also provides a more complete response and acknowledges its role as that of a medical assistant.
3. From a layman's point of view, this response looks even more complete than the one received with temperature = 0 parameter value.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(new_response(query,max_tokens=1024,temperature=0.5))

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The common symptoms for appendicitis include:
1. Sudden pain in the lower right abdomen, which may start as mild and then progress to severe over hours.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Vomiting.
4. Constipation or diarrhea.
5. Fever, which can be low-grade at first but can rise as high as 101 degrees Fahrenheit (38.3 degrees Celsius) if the appendix ruptures.
6. Abdominal swelling and tenderness.
7. Pain around the navel or the pain may shift to that area when you press on the lower right abdomen.
8. Inability to pass gas or have a bowel movement.
9. Feeling restless, unable to find a comfortable position.
10. Low-grade fever and chills.

Appendicitis cannot be cured via medicine alone. If the appendix ruptures, it can lead to peritonitis, a serious and potentially life-threatening condition. Therefore, when diagnosed

#Observations:
1. The response has one additional bullet point for the symptoms of appendicitis.
2.  It also confirms that the surgery is needed and medicine alone does not work.  In most of these respects, the response is almost identical to the one with temperature=0 parameter.
3.  From layman point of view, this answer is substantially similar to the one with low (more deterministic) value of the temperature parameter.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(new_response(query,max_tokens=1024,temperature=0.5))

Llama.generate: prefix-match hit



I'd be happy to help you with your question! However, I must clarify that I'm a medical assistant and not a doctor. Therefore, I can provide you with information based on available resources and research, but I cannot diagnose or prescribe treatments.

Regarding your question, sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp, face, or other parts of the body. The exact cause of this condition is not fully understood, but it's believed to be related to a problem with the immune system attacking hair follicles.

There are several treatments and solutions for addressing sudden patchy hair loss. Some of them include:

1. Corticosteroids: These medications can be applied directly to the affected area or taken orally to reduce inflammation and suppress the immune system's response against the hair follicles. This treatment may help promote hair regrowth.
2. Immunotherapy: Injections of certain substances, 

#Observations:
1.  The treatment adds lifestyle modifications as an important bullet point, but removes alternative therapies from the treatment response.
2.  It adds a  more comprehensive list of the causes to sudden patchy hair loss.
3.  Combining the two responses and evaluating with different termparture values is needed to get more comprehensive response and then validating it with domain experts.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(new_response(query,max_tokens=1024,temperature=0.5))

Llama.generate: prefix-match hit



Based on the information available to me, I would recommend several treatments for a person who has sustained a physical injury to brain tissue, leading to temporary or permanent impairment of brain function. 

First and foremost, immediate medical attention is crucial. Depending on the severity of the injury, this may involve emergency surgery, intensive care unit (ICU) admission, or close monitoring in a hospital setting. 

For acute treatment, medications such as diuretics, corticosteroids, and anticonvulsants may be prescribed to manage symptoms like swelling, inflammation, and seizures. Rehabilitation therapy is another essential aspect of care for brain injury patients. This includes physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation to help restore lost functions and improve overall quality of life. 

Additionally, alternative therapies like hyperbaric oxygen therapy, transcranial magnetic stimulation (TMS), and acupuncture may be considered as

#Observations:
1. The response is largely similar to the one previously provided.  However, it moves from a bullet point construct to a more free-writing paragraph kind of construct.
2. Personally, it feels like the previous response might be a bit more quicker to grasp for a doctor than this response as it is more free flowing, and does not categorize treatments in buckets.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(new_response(query,max_tokens=1024,temperature=0.5))

Llama.generate: prefix-match hit



The first step in managing a leg fracture that occurs during a hiking trip is to ensure the safety of both the injured person and any companions. If possible, try to prevent further injury by stabilizing the affected limb with a splint or improvised sling. This can help reduce pain and swelling while also preventing excessive movement that could worsen the fracture.

Once the injured person is stable, it's important to assess their overall condition, including checking for signs of shock such as rapid heartbeat, shallow breathing, or pale skin. If any signs of shock are present, immediate medical attention should be sought.

If the fracture is not severe and the person is stable, they can be transported back to civilization for further care. During transportation, make sure the injured leg is kept elevated as much as possible to reduce swelling and pain. It's also important to keep the leg immobilized with a splint or sling to prevent any movement that could worsen the fracture.

Upon

#Observations:
1. The response for this query also moves from a bullet point response to a free-flowing textual paragraph.
2.  While the points are largely similar, it does add a few things like X-rays and precautions such as not lifting weights which were missing in previous response.
3.  A combination of the two responses is best suited to provide a more comprehensive response to the query.

#Step 6 : Let us now reduce the top_p parameter from its default value of 0.95 for the response to be restricted to the top few key points.  

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query="What is the protocol of managing Sepsis in a critical care unit?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_p=0.1))

Llama.generate: prefix-match hit



I'd be happy to help answer your question regarding the protocol for managing sepsis in a critical care unit. The treatment for sepsis involves addressing its underlying causes while providing supportive care to maintain organ function and prevent complications. Here are some general steps that may be taken:

1. Early recognition and prompt initiation of antibiotic therapy: This is crucial as delaying antibiotic administration can worsen the condition and increase mortality. The choice of antibiotics depends on the suspected pathogen and local resistance patterns.

2. Fluid resuscitation: Sepsis often leads to hypovolemia, so intravenous fluids are administered to maintain adequate blood pressure and organ perfusion. Crystalloids like normal saline or lactated Ringer's solution are commonly used for initial resuscitation.

3. Vasopressor support: In cases of septic shock, where blood pressure remains low despite fluid resuscitation, vasopressors may be required to maintain adequate pe

#Observations:
1. This response combines the glucose control, electrolyte control, and followup steps into a single "prevention of secondary infection" step.  Thus, it chose to condense some of the specific treatments into a single bullet point.
2.  Domain expert might be needed to determine if one response can be considered better than the other.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_p=0.1))

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The common symptoms for appendicitis include:
1. Sudden pain in the lower right abdomen, which may begin as mild and then gradually worsens over several hours.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Vomiting.
4. Fever, which may be low-grade at first but can rise to very high temperatures as the condition progresses.
5. Constipation or diarrhea.
6. Abdominal swelling and tenderness.
7. Pain in the lower right quadrant of the abdomen when moving or pressing on that area.
8. Inability to pass gas or have a bowel movement.
9. Feeling restless or unable to find a comfortable position due to pain.

Appendicitis cannot be cured via medicine alone, as the inflammation and swelling of the appendix can obstruct the blood flow, leading to tissue death and potential rupture. If left untreated, an appendix rupture can lead to peritoniti

#Observations:
1. This response stays largely same as the previous one.  Reduction of top_p not changing the response seems to suggest that the source literature is similar or has similar meaning text across a large part of the sources.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_p=0.1))

Llama.generate: prefix-match hit



I'd be happy to help answer your question. However, I must first clarify that I am a medical assistant and not a doctor. Therefore, I cannot provide definitive medical advice or treatments. I can, however, share some information about possible causes and potential solutions for sudden patchy hair loss.

Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes the immune system to attack hair follicles, leading to hair loss in small patches on the scalp or other areas of the body. The exact cause of alopecia areata is not known, but it is believed to be related to genetics and environmental factors.

There are several treatments that may help address sudden patchy hair loss:

1. Corticosteroids: These medications can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally.
2. Minoxidil: This medication is a vasodilator that can help increase blood flow to the hair follicl

#Observations:
1. The response is largely similar, however the point about life-style modifications has been replaced with alternative therapies.  The continuous shift in these two points with slightly different prompts shows that one might want to consider using two different prompts to generate answers for this query.
2.  Other than that shift, the response being similar to previous query responses shows that the source text articles have similar treatments mentioned across a large range of publications on this topic.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_p=0.1))

Llama.generate: prefix-match hit



Based on the information available to me, I would recommend several treatments for a person who has sustained a physical injury to brain tissue, leading to temporary or permanent impairment of brain function. These treatments may include:

1. Medications: Depending on the specific symptoms and conditions, various medications may be prescribed to manage seizures, control pain, reduce inflammation, improve cognitive function, and address other complications.

2. Rehabilitation therapy: A combination of physical, occupational, speech, and cognitive rehabilitation therapies can help individuals regain lost skills and functions, improve overall functioning, and enhance their quality of life.

3. Surgery: In some cases, surgical intervention may be necessary to remove hematomas or other lesions that are causing pressure on the brain, alleviate seizures, or address other complications.

4. Assistive devices: Depending on the extent and nature of the injury, assistive devices such as wheelcha

#Observations:
1. The response goes back to being bulleted instead of free-flowing text.
2. It provides the same 7 bullet points, but removes the caution for consulting a doctor.
3.  This example does show that restricting the top-p value provides the most critical and relevant examples, but perhaps misses on some of the nuances of the response.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_p=0.1))

Llama.generate: prefix-match hit



A fractured leg is a serious injury that requires prompt medical attention. Here are some necessary precautions and treatment steps for a person with a fractured leg during a hiking trip:

1. **Stabilize the injury**: Try to prevent any further movement of the injured leg by using a splint, sling, or other immobilizing device. This will help reduce pain and prevent worsening of the injury.
2. **Provide first aid**: Apply a clean pad to the wound and apply gentle pressure with a sterile dressing to control bleeding. Splint the leg above and below the injury if possible.
3. **Transport the person safely**: If the person is unable to walk, use a stretcher or other carrying device to transport them down the mountain or trail safely. Avoid jolting or bouncing the injured leg during transportation.
4. **Seek medical help**: Once you have reached a medical facility, the healthcare professional will assess the severity of the fracture and determine the appropriate treatment. This may include 

#Observations:
1. The format of the response goes back to being bulleted.  However, it removes some of the nuances of using X-Rays and the precautions to be taken while getting treated for fracture.
2.  Again shows that restricting the top-p value provides the most critical and relevant examples, but perhaps misses on some of the nuances of the response.

#Step 7 : Let us now reduce the top_k parameter from its default value of 50 for the response to be restricted to the top few key points.  

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query="What is the protocol of managing Sepsis in a critical care unit?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_k=5))

Llama.generate: prefix-match hit



I am unable to provide an exact protocol as sepsis management can vary depending on the specific case and individual patient needs. However, I can outline some general steps that are commonly taken in managing sepsis in a critical care unit:

1. Early recognition and diagnosis: Sepsis should be suspected in any patient who is showing signs of infection and has organ dysfunction or evidence of inflammation.
2. Initial assessment and resuscitation: This includes administering oxygen, starting intravenous fluids, and providing vasopressors if necessary to maintain adequate blood pressure and organ perfusion.
3. Antibiotic therapy: Broad-spectrum antibiotics should be started as soon as possible based on the suspected source of infection and local antibiogram data.
4. Source control: Any identified sources of infection, such as an abscess or a catheter, should be addressed promptly to prevent further spread of infection.
5. Fluid management: Close monitoring of fluid balance is essential 

#Observations:
1. While the resposne loses a bit of finesse, it is able to provide the same 10 bullet points as have been shown in other responses.  It thus shows that this subject is exhaustively covered in some of the top relevance sources.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_k=5))

Llama.generate: prefix-match hit



Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The common symptoms of appendicitis include:
1. Abdominal pain, usually starting around the navel area and then shifting to the lower right side of the abdomen.
2. Loss of appetite
3. Nausea and vomiting
4. Fever (often above 101 degrees Fahrenheit or 38.3 degrees Celsius)
5. Diarrhea or constipation
6. Abdominal swelling and rigidity
7. Inability to pass gas or have a bowel movement

Appendicitis cannot be cured via medicine alone as the inflammation can lead to rupture of the appendix, which can result in peritonitis, a serious infection that requires immediate surgery. If left untreated, appendicitis can be life-threatening. The standard surgical procedure for treating appendicitis is an appendectomy, which involves removing the inflamed appendix through an incision in the abdomen. This procedure is typically performed as an emergency surgery.


#Observations:
1. The response removes a couple of the symptoms of appendicitis.  It also removes the options of surgery between standard and laproscopic.  However, in all other senses, the response is identical to the previous responses.
2.  Thus, the llm is able to pick the key critical sources and provide a comprehensive response using them.  Shows that the top_k parameter could be used to identify the most critical or relevant sources, and utiilzed to provide a less nuanced but most relevant answer.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_k=10))

Llama.generate: prefix-match hit



The possible causes of sudden patchy hair loss, also known as alopecia areata, include:
1. Autoimmune disorders - an abnormal response by your body's immune system that attacks your own hair follicles.
2. Stress or emotional shock - can trigger the onset of this condition.
3. Genetic factors - some people may be more prone to developing alopecia areata due to their family history.
4. Vitamin deficiencies - particularly in iron, zinc, and vitamin B12.
5. Hormonal imbalances - such as thyroid disorders or polycystic ovary syndrome (PCOS).

The effective treatments for addressing sudden patchy hair loss include:
1. Corticosteroids - injected directly into the bald spots to suppress the immune system's attack on the hair follicles.
2. Immunomodulators - drugs that help regulate the immune system and reduce inflammation, such as minoxidil or anthralin.
3. Hair transplant surgery - moving healthy hair from other parts of the scalp to the bald spots.
4. Wigs or hairpieces - a temporary solut

#Observations:
1. In this case, the response ends up being more comprehensive than any previous responses for this query.  It goes on to provide clear cases, as well as 8 different treatments for sudden patchy hair loss.
2. Once again shows that restricting top_k parameter could at times condense the response to top most relevant answers, while losing some of its nuance.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_k=5))

Llama.generate: prefix-match hit



Based on the information available to me, I would recommend several treatments for a person who has sustained a physical injury to brain tissue, leading to temporary or permanent impairment of brain function. These treatments may include:

1. Medications: Depending on the specific symptoms and conditions, various medications might be prescribed to manage seizures, control pain, reduce inflammation, improve cognitive function, and address other related issues.

2. Rehabilitation therapy: A combination of physical, occupational, speech, and cognitive rehabilitation therapies can help restore lost functions and improve overall quality of life. These therapies may include exercises to enhance motor skills, speech and language training, and cognitive stimulation activities.

3. Assistive devices: Depending on the extent of brain damage, assistive devices such as wheelchairs, communication aids, or prosthetic limbs might be necessary to help individuals regain independence and perform daily

Observations:
1.  The response is largely identical to the one with top_k value at 50.  Thus, it seems that the response was already picking most of the answer from the top relevant sources.
2. For this query, perhaps the additional sources do not add any additional meaning or value to the top sources of the search.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(new_response(query,max_tokens=1024,temperature=0.5,top_k=5))

Llama.generate: prefix-match hit



A fracture in the leg is a serious injury that requires prompt medical attention. Here are the necessary precautions and treatment steps for someone who has sustained a leg fracture while hiking:

1. Keep the person calm and still to prevent further injury or damage to the bone.
2. Apply a splint or immobilize the affected leg with a sling, bandage, or other supportive device to help maintain alignment and reduce pain.
3. If the fracture is severe or if the person is unable to walk, call for emergency medical assistance or transport the person to the nearest hospital as soon as possible.
4. Administer first aid measures such as cleaning and dressing any wounds, controlling bleeding, and providing pain relief with over-the-counter medications or ice packs.
5. Once the person has received medical attention, follow the doctor's instructions for care and recovery, which may include:
   a. Immobilizing the leg with a cast or brace to allow the bone to heal properly.
   b. Resting and avoid

#Observations:
1. This response is more comprehensive than most other responses previously.  It separates out treatment steps from the post treatment care, and details out the care steps as well.
2. While it covers almost all topics better than previous responses to this query, the point about using X-rays is missing perhaps beacause it is a very obvious step.

#Overall Observations and recommendations from the prompt engineering exercise:
1.  Increasing Max_Tokens to a number closer to 1000 in this case helps provide a more complete and exhaustive response.
2.  Increasing the temperature should be done only slightly as increasing it too much could result in free-flowing text in this case. Restricting it to 0.2 or 0.3 might be better than taking it to 0.5.
3.  Top_p and top_k values could be kept at their respective default scores of 0.95 and 50 to increase comprehensivenss or exhaustiveness of the response.

#QUESTION ANSWERING USING RAG

Let us now use external contextual data as a source to assess if it adds more specificity to the response.

#Loading the Data

In [ ]:
med_assistant_pdf_path = "medical_diagnosis_manual.pdf"

In [ ]:
pdf_loader = PyMuPDFLoader(med_assistant_pdf_path)

In [ ]:
med_assistant = pdf_loader.load()

#Data Overview
##Checking the first 3 pages

In [ ]:
for i in range(3):
    print(f"Page Number : {i+1}",end="\n")
    print(med_assistant[i].page_content,end="\n")

Page Number : 1
vishal.suri@gmail.com
NW2RKBYF0U
This file is meant for personal use by vishal.suri@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
vishal.suri@gmail.com
NW2RKBYF0U
This file is meant for personal use by vishal.suri@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...............................................................................................................................

#Checking the 6th page

In [ ]:
med_assistant[5].page_content

'1513\nChapter 145. Trematodes (Flukes)    ....................................................................................................................................\n1520\nChapter 146. Cestodes (Tapeworms)    ...............................................................................................................................\n1527\nChapter 147. Intestinal Protozoa    ........................................................................................................................................\n1536\nChapter 148. Extraintestinal Protozoa    .............................................................................................................................\n1555\nChapter 149. Viruses    ...............................................................................................................................................................\n1559\nChapter 150. Respiratory Viruses    ................................................................

#Checking the number of pages

In [ ]:
len(med_assistant)

4114

#Data Preparation for RAG

#Data Chunking

In [ ]:
#Utilizing character text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)

In [ ]:
#Create chunks based on character splitting
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
#Determine the total number of chunks
len(document_chunks)

8469

In [ ]:
#Determine the content of first chunk
document_chunks[0].page_content

'vishal.suri@gmail.com\nNW2RKBYF0U\nThis file is meant for personal use by vishal.suri@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
#Determine the content of second last chunk
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\nvishal.suri@gmail.com\nNW2RKBYF0U\nThis file is meant for personal use by vishal.suri@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
#Determine the content of the last chunk
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

#Embedding

In [ ]:
#Use sentence transformer from gte large as it is a high performance BERT based encoder
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

/tmp/ipykernel_2277/4198310515.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
#Determine embeddings of the first and second chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [ ]:
#Determine dimensions of the first embedding
print("Dimension of the embedding vector ",len(embedding_1))
#Ensure all embeddings are similar in vector size
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

#Vector Database

In [ ]:
#Create directory to store vector database
out_dir = 'medical_assistant_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
#Utilize Chroma which is an open source AI database to store, index, and query vectors
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [ ]:
#Initialize, Load the vector embeddings and meta data
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_2277/2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [ ]:
#print the embeddings
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
#Test the similarity search for the vector with some keywords, and get top 3 results
vectorstore.similarity_search("Pulmonary Embolism",k=3)

[Document(metadata={'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'moddate': '2026-03-04T11:39:32+00:00', 'file_path': 'medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2012-06-15T05:44:40+00:00', 'page': 2079, 'total_pages': 4114, 'author': '', 'trapped': '', 'modDate': 'D:20260304113932Z', 'keywords': '', 'creationDate': 'D:20120615054440Z', 'format': 'PDF 1.7', 'source': 'medical_diagnosis_manual.pdf', 'subject': '', 'creator': 'Atop CHM to PDF Converter'}, page_content='Chapter 194. Pulmonary Embolism\nIntroduction\nPulmonary embolism (PE) is the occlusion of ≥ 1 pulmonary arteries by thrombi that originate\nelsewhere, typically in the large veins of the lower extremities or pelvis. Risk factors are\nconditions that impair venous return, conditions that cause endothelial injury or dysfunction,\nand underlying hypercoagulable states. Symptoms are nonspecific and include dyspnea,\npleuritic chest pain, cou

#Retriever

In [ ]:
#Define the retriever from vector database with a default of top 2 results
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [ ]:
#Test the working of retreiver on some questions
rel_docs = retriever.get_relevant_documents("Can you provide the trade names of medications used for treating hypertension?")
rel_docs

/tmp/ipykernel_1811/496813956.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("Can you provide the trade names of medications used for treating hypertension?")


[Document(metadata={'subject': '', 'modDate': 'D:20260304113932Z', 'moddate': '2026-03-04T11:39:32+00:00', 'page': 3677, 'keywords': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'format': 'PDF 1.7', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'trapped': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'author': '', 'total_pages': 4114, 'creationDate': 'D:20120615054440Z', 'source': 'medical_diagnosis_manual.pdf', 'creator': 'Atop CHM to PDF Converter', 'file_path': 'medical_diagnosis_manual.pdf'}, page_content='Appendix III: Trade Names of Some Commonly Used Drugs\nThroughout THE MANUAL, generic (nonproprietary) names for drugs are used whenever possible. Most\nprescription drugs have trade names (also called proprietary, brand, or specialty names) to distinguish\nthem as being produced and marketed by a particular manufacturer. In the US, these names are usually\nregistered as trademarks with the Patent Office, which confers certain legal right

#Defining the Response Generator

The response generator can be the same LLM llama used earlier

#System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

1. The system message describing the assistant's role.
2. A user message template including context and the question.

In [ ]:
qna_system_message_rag = """
You are a medical assistant whose work is to review the report and provide the appropriate answers to a medical doctor from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Answer only using the context provided in the input. Do not mention anything about the context in your final answer.

To the extent possible, answer using bullet points for individual points instead of free flowing text.

If the answer is not found in the context, do not make up an answer, Do not hallucinate, instead Respond "I don't know".
"""

In [ ]:
qna_user_message_rag = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

#Response Function

In [ ]:
#Define a function that combines system and user messages for RAG and outputs the final generated response from the llm
def generate_rag_response(user_input,k=3,max_tokens=1024,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message_rag,qna_user_message_rag
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    print("context list is as follows:\n",context_list,"\n\n")
    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_rag.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message_rag + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k,
                  stop=["</s>"],echo=False
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit ?"
print(generate_rag_response(user_input))

context list is as follows:
 ["16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high\nnurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring\nof physiologic parameters.\nSupportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of\ninfection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to\n25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help\ndying patients maintain dignity (see p. 3480).\nPatient Monitoring and Testing\nSome monitoring is manual (ie

Llama.generate: prefix-match hit


Answer:
- Broad-spectrum antibiotics immediately for patients with suspected serious bacterial infections or acute chest syndrome
- Liberal administration of analgesics, usually opioids for painful crises
- Maintaining normal intravascular volume
- Transfusion is given in many situations but not during an uncomplicated painful crisis. Indications include prevention of recurrent cerebral thrombosis, especially in children, acute splenic sequestration, aplastic crises, cardiopulmonary symptoms or signs, preoperative use, priapism, and life-threatening events that would benefit from improved O2 delivery.


#Observations:
1. This response is far more specific and to-the-point compared to the generic responses that we saw from the llm earlier.
2.  However, it could be that the response is less comprehensive than the one we had with llm when it was perhaps searching through far more documents.
3.  Increasing the k value should result in more comprehensive response.  

4.  Increasing temperature could result in slightly more comprehensive response.  
5.  The response does look relevant to the query, and potentially taken from the context.
6.  Review by Domain expert will help establish comprehensiveness, relevance and accuracy of the response.

7.  Validation check for groundness and relevance will help establish the overall validity of the response.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine?  If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2)

context list is as follows:
 ["• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can\nusually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these\ncases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to\nappendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis\nand a normal cecum, the appendix should be removed.\nAppendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For\nnonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics\nshould be continued until the patient's temperature and WBC count have normalized or continued for a\nfixed course, according to the surge

Llama.generate: prefix-match hit


'- Appendicitis symptoms: abdominal pain, anorexia (loss of appetite), and abdominal tenderness.\n- No, appendicitis cannot be cured via medicine alone. It requires surgical removal of the appendix.'

Observations:
1.  Again, a very specific - perhaps too specific response compared to what we had from open llm's.
2.  While the response addresses both the questions, the answers seem too short and to-the-point.  They seem to miss the details, the nuances, and the comprehensiveness that we saw from llm with prompt engineering.
3.  Increasing the context sources (k value), and temperature (for non-deterministic answers) should improve the comprehensivenss of the response.
4.  Using an LLM to check the groundedness and relevance; using a domain expert will both help in reviewing and improving the accuracy of the response.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3)

context list is as follows:
 ['corticosteroids, retinoids, or immunosuppressants.\nHair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be\ndifferent in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium\nis usually temporary as well and abates after the precipitating agent is eliminated.\nKey Points\n• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.\n• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the\nunderlying disorder.\n• Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.\nAlopecia Areata\nAlopecia areata is sudden patchy hair loss in people with no obvious skin or systemic disorder.\nThe scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may\naffect most or all of the body (alopecia universalis). Alopecia ar

Llama.generate: prefix-match hit


'- Sudden patchy hair loss can be a symptom of alopecia areata, an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.\n- Treatment options for alopecia areata include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA).\n- Hormonal modulators such as oral contraceptives or spironolactone may be useful for female-pattern hair loss associated with hyperandrogenemia.\n- Surgical options include follicle transplant, scalp flaps, and alopecia reduction.\n- Other causes of sudden patchy hair loss should be treated accordingly. For example, tinea capitis can be treated with topical or oral antifungals, trichotillomania with behavior modification, clomipramine, or an SSRI, and scarring alopecia with a long-acting oral tetracycline in combination with a potent topical corticosteroid.\n- Scalp biopsy 

#Observations:
1. This response definitely seems much more specific and comprehensive compared to the responses generated earlier.  
2.  It goes into great details about this ailment from the source documents.
3.  However, it was not able to succinctly summarize the entire set of sources into neatly categorized bullet points.
4.  Need to check with LLM about the groundedness and relevance of the response.
5.  Perhaps improving k-value and reducing max_tokens could help in this case to make the response more comprehensive and to-the-point.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4)

context list is as follows:
 ['Chapter 324. Traumatic Brain Injury\nIntroduction\nTraumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently\nimpairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily\nCT). Initial treatment consists of ensuring a reliable airway and maintaining adequate\nventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more\nsevere injury to place monitors to track and treat intracranial pressure, decompress the brain if\nintracranial pressure is increased, or remove intracranial hematomas. In the first few days after\nthe injury, maintaining adequate brain perfusion and oxygenation and preventing complications\nof altered sensorium are important. Subsequently, many patients require rehabilitation.\nIn the US, as in much of the world, TBI is a common cause of death and disability. Causes include motor\nvehicle crashes and other transportation-related causes (e

Llama.generate: prefix-match hit


'Answer:\n- Ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure\n- Surgery if needed for monitoring intracranial pressure, decompression, or hematoma removal\n- Preventing complications in the first few days after injury\n- Maintaining adequate brain perfusion and oxygenation\n- Rehabilitation subsequently.'

#Observations:
1. The response matches the generic response produced by the llm with prompt engineering.
2.  The specific response comes across as a set of instructions, and is more pointed.  It is perhaps more relatable to a medical doctor than the one provided by previous LLM exercise.
3.  It covers all the key aspects demanded from the LLM - bullet points; specificity; summarize all key points.  Increase in k-value could improve number of sources at its disposal for creating better responses.
4.  Review by another llm could help validate the greoundedness and relevance of the response.  As such, knowing the answers to the same question from previous LLM exercises, the response does look relevant to the query.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5)

context list is as follows:
 ["batting, and layers 2 and 4 are elastic bandages. The injured limb is elevated above the heart for the first\n2 days in a position that allows gravity to help drain edema fluid and thus minimize swelling. After 48 h,\nperiodic application of warmth (eg, a heating pad) for 15 to 20 min may relieve pain and speed healing.\nImmobilization: Immobilization decreases pain and facilitates healing by preventing further injury and is\nhelpful except for very rapidly healing injuries. Joints proximal and distal to the injury should be\nimmobilized.\nA cast is usually used for fractures or other injuries that require weeks of immobilization. Rarely, swelling\nunder a cast is severe enough to contribute to compartment syndrome (see p. 3213). Sometimes, if\nsevere swelling is likely, a cast (and all padding) is cut open from end to end medially and laterally\n(bivalved). Patients with casts should be given written instructions:\n• To keep the cast dry\n• Never to put 

Llama.generate: prefix-match hit


'- Immobilize the injured limb using an elastic bandage or a cast.\n  - Elevate the limb above heart level for first 2 days to minimize swelling.\n  - Apply warmth after 48 hours to relieve pain and speed healing.\n  - Keep the cast dry and never put any object inside it.\n  - Inspect the skin around the cast daily, applying lotion to any red or sore areas.\n  - Pad any rough edges with soft material to prevent discomfort.\n  - Seek medical care if an odor emanates from within the cast or a fever develops.\n- Immobilize the injury using a splint for stable injuries that require less immobilization time.\n  - Allow patients to apply ice and move more, reducing the risk of compartment syndrome.\n- Consider prolonged bed rest for fractures that require it, but be aware of potential complications such as deep venous thrombosis and urinary tract infections.\n- Encourage early mobilization for rapidly healing injuries to minimize contractures and muscle atrophy.\n- For frostbitten extremitie

#Observations:
1. The response is much more specific to the query and covers more ground than the generic llm did.  It responds much more to the specific treatment and care post fracture.  However, it does not pay much attention to the fracture occuring due to hiking as the key environment.
2.  Fine tuning the chunking, retreival, and llm parameters could improve the response further.

# Fine-tuning

Fine tuning can be performed through tuning chunking, retrieval, and llm parameters.

#Step 1 Let us start with fine tuning chunking parameters.

###Let us increase the chunk size, and increase the overlap in the chunks so that bigger data-size can be covered in the source data.

In [ ]:
#Increase chunk size and chunk overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1024,
    chunk_overlap= 40
)

In [ ]:
#Carry out the chunking again
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
#Determine the new lower number of chunks
len(document_chunks)

4545

In [ ]:
#Compute the new embeddings as an example
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [ ]:
#Ensure that the embedding vector length stays the same for all chunks
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

In [ ]:
#Define the new directory for the altered chunks and their embeddings
out_dir = 'medical_assistant_db2'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
#Create the new vector store for these embeddings using Chroma
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [ ]:
#Initialize the new database
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

In [ ]:
#Print the new embeddings
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

We shall keep the retreiving and llm parameters same for now to isolate and assess the impact of higher chunking size alone.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit ?"
print(generate_rag_response(user_input))

context list is as follows:
 ["16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high\nnurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring\nof physiologic parameters.\nSupportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of\ninfection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to\n25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help\ndying patients maintain dignity (see p. 3480).\nPatient Monitoring and Testing\nSome monitoring is manual (ie

Llama.generate: prefix-match hit


###Answer
- Broad-spectrum antibiotics immediately for patients with suspected serious bacterial infections or acute chest syndrome
- Liberal administration of analgesics, usually opioids for painful crises
- Maintaining normal intravascular volume
- Transfusion is given in many situations such as prevention of recurrent cerebral thrombosis, especially in children, when Hb is < 5 g/dL, during life-threatening events that would benefit from improved O2 delivery, and for cardiopulmonary symptoms or signs. However, it is not helpful during an uncomplicated painful crisis.


#Observations:
1. The response is almost identical to the one with smaller chunk size.  This implies that it is able to get the narrative without increasing chunk size.  The response continues to be precise and specific to the question asked.
2.  Increasing the retrival source documents or increasing the llm temperature could lead to more descriptive response.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine?  If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2)

context list is as follows:
 ["• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can\nusually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these\ncases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to\nappendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis\nand a normal cecum, the appendix should be removed.\nAppendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For\nnonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics\nshould be continued until the patient's temperature and WBC count have normalized or continued for a\nfixed course, according to the surge

Llama.generate: prefix-match hit


'- Appendicitis symptoms: abdominal pain, anorexia (loss of appetite), and abdominal tenderness.\n- No, appendicitis cannot be cured via medicine alone. It requires surgical removal of the appendix.'

#Observations:
1.  The response continues to be identical to the vectorization with smaller chunk size.  No change in the narrative with this source either.
2.  Increasing the number of documents retreived and increasing the temperature could potentially make the narrative richer.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3)

context list is as follows:
 ['corticosteroids, retinoids, or immunosuppressants.\nHair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be\ndifferent in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium\nis usually temporary as well and abates after the precipitating agent is eliminated.\nKey Points\n• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.\n• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the\nunderlying disorder.\n• Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.\nAlopecia Areata\nAlopecia areata is sudden patchy hair loss in people with no obvious skin or systemic disorder.\nThe scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may\naffect most or all of the body (alopecia universalis). Alopecia ar

Llama.generate: prefix-match hit


'- Sudden patchy hair loss can be a symptom of alopecia areata, an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.\n- Treatment options for alopecia areata include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA).\n- Hormonal modulators such as oral contraceptives or spironolactone may be useful for female-pattern hair loss associated with hyperandrogenemia.\n- Surgical options include follicle transplant, scalp flaps, and alopecia reduction.\n- Other causes of sudden patchy hair loss should be treated accordingly. For example, tinea capitis can be treated with topical or oral antifungals, trichotillomania with behavior modification, clomipramine, or an SSRI, and scarring alopecia with a long-acting oral tetracycline in combination with a potent topical corticosteroid.\n- Scalp biopsy 

#Observations:
1. Once again, the response here is identical to the one with smaller chunk sizes.
2.  It indicates that the embedding model is robust enough to locate the core semantic answer regardless of surrounding context density. It suggests that the crucial information relevant to query is sufficiently distinctive and self-contained, making the additional context in larger chunks or reduced context in smaller chunks irrelevant to the final output.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4)

context list is as follows:
 ['Chapter 324. Traumatic Brain Injury\nIntroduction\nTraumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently\nimpairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily\nCT). Initial treatment consists of ensuring a reliable airway and maintaining adequate\nventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more\nsevere injury to place monitors to track and treat intracranial pressure, decompress the brain if\nintracranial pressure is increased, or remove intracranial hematomas. In the first few days after\nthe injury, maintaining adequate brain perfusion and oxygenation and preventing complications\nof altered sensorium are important. Subsequently, many patients require rehabilitation.\nIn the US, as in much of the world, TBI is a common cause of death and disability. Causes include motor\nvehicle crashes and other transportation-related causes (e

Llama.generate: prefix-match hit


'Answer:\n- Ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure\n- Surgery if needed for monitoring intracranial pressure, decompression, or hematoma removal\n- Preventing complications in the first few days after injury\n- Maintaining adequate brain perfusion and oxygenation\n- Rehabilitation subsequently.'

#Observations:
1. For this query also, the response here is identical to the one with smaller chunk sizes.
2.  It indicates that the embedding model is robust enough to locate the core semantic answer regardless of surrounding context density. It suggests that the crucial information relevant to query is sufficiently distinctive and self-contained, making the additional context in larger chunks or reduced context in smaller chunks irrelevant to the final output.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5)

context list is as follows:
 ["batting, and layers 2 and 4 are elastic bandages. The injured limb is elevated above the heart for the first\n2 days in a position that allows gravity to help drain edema fluid and thus minimize swelling. After 48 h,\nperiodic application of warmth (eg, a heating pad) for 15 to 20 min may relieve pain and speed healing.\nImmobilization: Immobilization decreases pain and facilitates healing by preventing further injury and is\nhelpful except for very rapidly healing injuries. Joints proximal and distal to the injury should be\nimmobilized.\nA cast is usually used for fractures or other injuries that require weeks of immobilization. Rarely, swelling\nunder a cast is severe enough to contribute to compartment syndrome (see p. 3213). Sometimes, if\nsevere swelling is likely, a cast (and all padding) is cut open from end to end medially and laterally\n(bivalved). Patients with casts should be given written instructions:\n• To keep the cast dry\n• Never to put 

Llama.generate: prefix-match hit


'- Immobilize the injured limb using an elastic bandage or a cast.\n  - Elevate the limb above heart level for first 2 days to minimize swelling.\n  - Apply warmth after 48 hours to relieve pain and speed healing.\n  - Keep the cast dry and never put any object inside it.\n  - Inspect the skin around the cast daily, applying lotion to any red or sore areas.\n  - Pad any rough edges with soft material to prevent discomfort.\n  - Seek medical care if an odor emanates from within the cast or a fever develops.\n- Immobilize the injury using a splint for stable injuries that require less immobilization time.\n  - Allow patients to apply ice and move more, reducing the risk of compartment syndrome.\n- Consider prolonged bed rest for fractures that require it, but be aware of potential complications such as deep venous thrombosis and urinary tract infections.\n- Encourage early mobilization for rapidly healing injuries to minimize contractures and muscle atrophy.\n- For frostbitten extremitie

#Observations:

1.  Same as all the observations above; increasing the chunking size alone is producing identical response here. It showcases the robustness of the embedding model and the distinctiveness of the query related information causing chunk size to be irrelevant between these two sizes.

#Step 2 Let us revert to the original chunk size and overlap.  This time, we increase the number of top relevant documents retrieved from 2 to 10.

In [ ]:
#Define out directory as the original vector database with previous smaller chunks
out_dir = 'medical_assistant_db'

In [ ]:
#Increase the search window to top 10 semantically similar results
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 10}
)

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit ?"
print(generate_rag_response(user_input,k=10))

context list is as follows:
 ["16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high\nnurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring\nof physiologic parameters.\nSupportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of\ninfection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to\n25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help\ndying patients maintain dignity (see p. 3480).\nPatient Monitoring and Testing\nSome monitoring is manual (ie

#Observations:

1.  Need to increase the context window of the llm to incorporate these many tokens for retreival.

In [ ]:
#Increase the context window to 7000 tokens instead of the default 5000 tokens
llm = Llama(
    model_path=model_path,
    n_ctx=7000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


Now, repeat the Query 1

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit ?"
print(generate_rag_response(user_input,k=10))

context list is as follows:
 ["16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high\nnurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring\nof physiologic parameters.\nSupportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of\ninfection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to\n25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help\ndying patients maintain dignity (see p. 3480).\nPatient Monitoring and Testing\nSome monitoring is manual (ie

Observations:
1. This resposne does seem like a step-by-step set of instructions that are very prescriptive.
2.  It matches with the broad direction shown in the earlier responses about broad-spectrum antiboiotics and analgesics for pain.  
3.  However, it does not talk of adminstering fluids and a few other steps.
4.  Need to check from domain experts if this response could be considered superior to the previous response for 2-relevant document reterival.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine?  If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,k=10)

context list is as follows:
 ["• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can\nusually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these\ncases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to\nappendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis\nand a normal cecum, the appendix should be removed.\nAppendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For\nnonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics\nshould be continued until the patient's temperature and WBC count have normalized or continued for a\nfixed course, according to the surge

Llama.generate: prefix-match hit


"- The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever (rectal temperature 37.7 to 38.3° C [100 to 101° F]) is common.\n- Appendicitis cannot be cured via medicine alone, and surgery is required for treatment. The surgical procedure for treating appendicitis is an appendectomy, which i

#Observations:

1.  This response is once again superior to any of the other responses we have seen before.  It is very prescriptive in its description of symptoms, as well as in the defining medicine versus surgery, and the kind of surgery needed.  
2.  The response goes onto describe the details of the surgery needed and procedures needed before and after the surgery as well in good detail.
3.  Increasing the context window and fetching / retreiving more search results has definitely improved the quality of response substantially.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,k=10)

context list is as follows:
 ['corticosteroids, retinoids, or immunosuppressants.\nHair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be\ndifferent in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium\nis usually temporary as well and abates after the precipitating agent is eliminated.\nKey Points\n• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.\n• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the\nunderlying disorder.\n• Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.\nAlopecia Areata\nAlopecia areata is sudden patchy hair loss in people with no obvious skin or systemic disorder.\nThe scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may\naffect most or all of the body (alopecia universalis). Alopecia ar

Llama.generate: prefix-match hit


'Based on the context provided, the condition being described is Alopecia Areata. The effective treatments for this condition include:\n- Corticosteroids (triamcinolone acetonide suspension or potent topical corticosteroids)\n- Topical anthralin\n- Minoxidil\n- Induction of allergic contact dermatitis using diphencyprone or squaric acid dibutylester\n\nThe possible causes behind sudden patchy hair loss, as described in the context, include an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. Other potential causes include tinea capitis, trichotillomania, discoid lupus, and secondary syphilis. However, measures of thyroid-stimulating hormone, vitamin B12, and autoantibodies are only indicated when coexisting disease is suspected.'

#Observations:
1.  This is once again a good outline of the symptoms and treatments that one could consider for patchy hair loss.
2.  The overall response seems relevant to both the context as well as to the query asked.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,k=10)

context list is as follows:
 ['Chapter 324. Traumatic Brain Injury\nIntroduction\nTraumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently\nimpairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily\nCT). Initial treatment consists of ensuring a reliable airway and maintaining adequate\nventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more\nsevere injury to place monitors to track and treat intracranial pressure, decompress the brain if\nintracranial pressure is increased, or remove intracranial hematomas. In the first few days after\nthe injury, maintaining adequate brain perfusion and oxygenation and preventing complications\nof altered sensorium are important. Subsequently, many patients require rehabilitation.\nIn the US, as in much of the world, TBI is a common cause of death and disability. Causes include motor\nvehicle crashes and other transportation-related causes (e

Llama.generate: prefix-match hit


'- Maintaining adequate brain perfusion and oxygenation\n- Preventing complications of altered sensorium\n- Supportive care (preventing systemic complications due to immobilization, providing good nutrition, preventing pressure ulcers)\n- Surgery for patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas\n- Rehabilitation for many patients subsequently\n- No specific treatment, but supportive care should include preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers\n- Drugs are given to prevent seizures if head injury causes significant structural injury or a Glasgow Coma Scale score of < 10\n- Long-term treatment involves multiple drugs for some patients and determining the appropriate dose based on patient tolerance and response\n- Corticosteroids are not recommended for controlling intracranial 

#Observations:

1.  The response is once again very comprehensive, and probably the best response that we have encountered so far.  
2.  It not only provides a good summary, it covers different angles well, and is not a generic answer.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,k=10)

context list is as follows:
 ["batting, and layers 2 and 4 are elastic bandages. The injured limb is elevated above the heart for the first\n2 days in a position that allows gravity to help drain edema fluid and thus minimize swelling. After 48 h,\nperiodic application of warmth (eg, a heating pad) for 15 to 20 min may relieve pain and speed healing.\nImmobilization: Immobilization decreases pain and facilitates healing by preventing further injury and is\nhelpful except for very rapidly healing injuries. Joints proximal and distal to the injury should be\nimmobilized.\nA cast is usually used for fractures or other injuries that require weeks of immobilization. Rarely, swelling\nunder a cast is severe enough to contribute to compartment syndrome (see p. 3213). Sometimes, if\nsevere swelling is likely, a cast (and all padding) is cut open from end to end medially and laterally\n(bivalved). Patients with casts should be given written instructions:\n• To keep the cast dry\n• Never to put 

Llama.generate: prefix-match hit


'Based on the context provided, here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:\n\n1. Immobilize the injured limb to prevent further injury and facilitate healing. This can be done using a splint or a cast, depending on the severity of the fracture and the length of immobilization required.\n2. Elevate the injured limb above heart level for the first 48 hours to help drain edema fluid and minimize swelling.\n3. Apply ice to the affected area intermittently during the first 24 to 48 hours to reduce swelling, inflammation, and pain. Use a plastic bag or towel to enclose the ice to prevent direct contact with the skin.\n4. Provide pain relief using analgesics such as acetaminophen or NSAIDs. If pain persists for more than 72 hours after the injury, referral to a specialist is recommended.\n5. Avoid putting any objects inside the cast and keep it dry to prevent infection and skin irritation.\n6. Inspect the edges of the 

#Observations:
1.  Once again, this is the most comprehensive and prescriptive step-by-step response we have received for this query.  While it is quite detailed, medical practiotioners might appreciate the thoroughness of this response over a very short and snappy response.

2.  Increasing the k-value reduces the performance of the system as each of the responses have become slower to fetch.

3.  However, the comprehensiveness of the response has increased due to inclusion of documents that would have otherwise been considered semantically less similar by the embedding document.

#Step 3 This time we keep the original chunk size.  We increase the number of top relevant documents retrieved to 10.  However, we reduce the max_tokens to 512 to assess if that reduces response time and improves succinctness of response.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit ?"
print(generate_rag_response(user_input,k=10,max_tokens=512))

context list is as follows:
 ["16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high\nnurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring\nof physiologic parameters.\nSupportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of\ninfection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to\n25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help\ndying patients maintain dignity (see p. 3480).\nPatient Monitoring and Testing\nSome monitoring is manual (ie

Llama.generate: prefix-match hit


Answer:
- Admit the patient to an ICU
- Begin IV hydration with 0.9% saline solution
- Consider cooling techniques such as evaporative cooling or ice water immersion
- Monitor for signs of organ dysfunction and rhabdomyolysis
- Administer antibiotics such as cefotaxime for at least 5 days and until ascitic fluid shows < 250 PMNs/μL
- Consider prophylactic antibiotics to prevent recurrence within a year
- Monitor for signs of peritoneal infection and perform diagnostic paracentesis if necessary
- Provide supportive care including management of pain, infection, and undernutrition
- Consider adjunctive therapy or surgery in some cases.


#Observations:
1.  The response is identical to the one with max_token size of 1024.  The bullet point construct has allowed the response to stay within limits despite it being quite thorough.
2.  It took about the same time as the other response as well.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine?  If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,k=10,max_tokens=512)

context list is as follows:
 ["• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can\nusually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these\ncases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to\nappendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis\nand a normal cecum, the appendix should be removed.\nAppendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For\nnonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics\nshould be continued until the patient's temperature and WBC count have normalized or continued for a\nfixed course, according to the surge

Llama.generate: prefix-match hit


"- The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever (rectal temperature 37.7 to 38.3° C [100 to 101° F]) is common.\n- Appendicitis cannot be cured via medicine alone, and surgery is required for treatment. The surgical procedure for treating appendicitis is an appendectomy, which i

Observations:
1.  The response is once again almost identical to that produced from max_tokens=1024 option.
2.  The execution took about the same time as that of the llm with token-size of 1024 tokens.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,k=10,max_tokens=512)

/tmp/ipykernel_2277/3752703135.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)


context list is as follows:
 ['corticosteroids, retinoids, or immunosuppressants.\nHair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be\ndifferent in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium\nis usually temporary as well and abates after the precipitating agent is eliminated.\nKey Points\n• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.\n• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the\nunderlying disorder.\n• Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.\nAlopecia Areata\nAlopecia areata is sudden patchy hair loss in people with no obvious skin or systemic disorder.\nThe scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may\naffect most or all of the body (alopecia universalis). Alopecia ar

'Based on the context provided, the condition being described is Alopecia Areata. The effective treatments for this condition include:\n- Corticosteroids (triamcinolone acetonide suspension or potent topical corticosteroids)\n- Topical anthralin\n- Minoxidil\n- Induction of allergic contact dermatitis using diphencyprone or squaric acid dibutylester\n\nThe possible causes behind sudden patchy hair loss, as described in the context, include an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. Other potential causes include tinea capitis, trichotillomania, discoid lupus, and secondary syphilis. However, measures of thyroid-stimulating hormone, vitamin B12, and autoantibodies are only indicated when coexisting disease is suspected.'

#Observations:
1.  The response is once again almost identical to that produced from max_tokens=1024 option.
2.  The execution took about the same time as that of the llm with token-size of 1024 tokens.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,k=10,max_tokens=512)

context list is as follows:
 ['Chapter 324. Traumatic Brain Injury\nIntroduction\nTraumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently\nimpairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily\nCT). Initial treatment consists of ensuring a reliable airway and maintaining adequate\nventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more\nsevere injury to place monitors to track and treat intracranial pressure, decompress the brain if\nintracranial pressure is increased, or remove intracranial hematomas. In the first few days after\nthe injury, maintaining adequate brain perfusion and oxygenation and preventing complications\nof altered sensorium are important. Subsequently, many patients require rehabilitation.\nIn the US, as in much of the world, TBI is a common cause of death and disability. Causes include motor\nvehicle crashes and other transportation-related causes (e

Llama.generate: prefix-match hit


'- Maintaining adequate brain perfusion and oxygenation\n- Preventing complications of altered sensorium\n- Supportive care (preventing systemic complications due to immobilization, providing good nutrition, preventing pressure ulcers)\n- Surgery for patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas\n- Rehabilitation for many patients subsequently\n- No specific treatment, but supportive care should include preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers\n- Drugs are given to prevent seizures if head injury causes significant structural injury or a Glasgow Coma Scale score of < 10\n- Long-term treatment involves multiple drugs for some patients and determining the appropriate dose based on patient tolerance and response\n- Corticosteroids are not recommended for controlling intracranial 

#Observations:
1.  Once again, the response is identical to the response from max_tokens at 1024.  Potentially, the response will become more succinct at a lower restriction on max_tokens.
2.  In this case, the execution was faster than the time taken in the response for max_tokens at 1024.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,k=10,max_tokens=512)

context list is as follows:
 ["batting, and layers 2 and 4 are elastic bandages. The injured limb is elevated above the heart for the first\n2 days in a position that allows gravity to help drain edema fluid and thus minimize swelling. After 48 h,\nperiodic application of warmth (eg, a heating pad) for 15 to 20 min may relieve pain and speed healing.\nImmobilization: Immobilization decreases pain and facilitates healing by preventing further injury and is\nhelpful except for very rapidly healing injuries. Joints proximal and distal to the injury should be\nimmobilized.\nA cast is usually used for fractures or other injuries that require weeks of immobilization. Rarely, swelling\nunder a cast is severe enough to contribute to compartment syndrome (see p. 3213). Sometimes, if\nsevere swelling is likely, a cast (and all padding) is cut open from end to end medially and laterally\n(bivalved). Patients with casts should be given written instructions:\n• To keep the cast dry\n• Never to put 

Llama.generate: prefix-match hit


'Based on the context provided, here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:\n\n1. Immobilize the injured limb to prevent further injury and facilitate healing. This can be done using a splint or a cast, depending on the severity of the fracture and the length of immobilization required.\n2. Elevate the injured limb above heart level for the first 48 hours to help drain edema fluid and minimize swelling.\n3. Apply ice to the affected area intermittently during the first 24 to 48 hours to reduce swelling, inflammation, and pain. Use a plastic bag or towel to enclose the ice to prevent direct contact with the skin.\n4. Provide pain relief using analgesics such as acetaminophen or NSAIDs. If pain persists for more than 72 hours after the injury, referral to a specialist is recommended.\n5. Avoid putting any objects inside the cast and keep it dry to prevent infection and skin irritation.\n6. Inspect the edges of the 

#Observations:
1.  The response is almost identical to that with max_tokens at 1024, with only a few words missing from the final sentence.

2.  Execution time was about the same as that in the instance with 1024 tokens.

#Step 4 This time we keep the original chunk size.  We increase the number of top relevant documents retrieved to 10.  However, we reduce the max_tokens to 256 to assess if that reduces response time and improves succinctness of response.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit ?"
print(generate_rag_response(user_input,k=10,max_tokens=256))

context list is as follows:
 ["16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high\nnurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring\nof physiologic parameters.\nSupportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of\ninfection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to\n25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help\ndying patients maintain dignity (see p. 3480).\nPatient Monitoring and Testing\nSome monitoring is manual (ie

Llama.generate: prefix-match hit


Answer:
- Admit the patient to an ICU
- Begin IV hydration with 0.9% saline solution
- Consider cooling techniques such as evaporative cooling or ice water immersion
- Monitor for signs of organ dysfunction and rhabdomyolysis
- Administer antibiotics such as cefotaxime for at least 5 days and until ascitic fluid shows < 250 PMNs/μL
- Consider prophylactic antibiotics to prevent recurrence within a year
- Monitor for signs of peritoneal infection and perform diagnostic paracentesis if necessary
- Provide supportive care including management of pain, infection, and undernutrition
- Consider adjunctive therapy or surgery in some cases.


#Observations:
1.  The response is identical to the one with max_tokens at 1024 or 512, however the time taken was about 30-40% less.  
2.  Thus, this response is more efficient and has the same comprehensiveness as the previous responses with more number of allowable tokens.
3.  The model likely encounters an EOS sequence flag at this point, so keeping the token size at 256 keeps the model efficient.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine?  If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,k=10,max_tokens=256)

context list is as follows:
 ["• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can\nusually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these\ncases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to\nappendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis\nand a normal cecum, the appendix should be removed.\nAppendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For\nnonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics\nshould be continued until the patient's temperature and WBC count have normalized or continued for a\nfixed course, according to the surge

Llama.generate: prefix-match hit


"- The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever (rectal temperature 37.7 to 38.3° C [100 to 101° F]) is common.\n- Appendicitis cannot be cured via medicine alone, and surgery is required for treatment. The surgical procedure for treating appendicitis is an appendectomy, which i

#Observations:
1.  While the execution again was faster, this response is condensed without it being summarized.  Thus, the response is less comprehensive in its semantics than the response with 1024 or 512 allowable tokens.
2.  The response describes the symptoms, and the kind of surgery but it does not go into details of the appendicitis surgery and its different nuances.  To a layman, it feels like the previous response would be appreaciated more by a medical professional for its thoroughness than this one.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,k=10,max_tokens=256)

context list is as follows:
 ['corticosteroids, retinoids, or immunosuppressants.\nHair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be\ndifferent in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium\nis usually temporary as well and abates after the precipitating agent is eliminated.\nKey Points\n• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.\n• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the\nunderlying disorder.\n• Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.\nAlopecia Areata\nAlopecia areata is sudden patchy hair loss in people with no obvious skin or systemic disorder.\nThe scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may\naffect most or all of the body (alopecia universalis). Alopecia ar

Llama.generate: prefix-match hit


'Based on the context provided, the condition being described is Alopecia Areata. The effective treatments for this condition include:\n- Corticosteroids (triamcinolone acetonide suspension or potent topical corticosteroids)\n- Topical anthralin\n- Minoxidil\n- Induction of allergic contact dermatitis using diphencyprone or squaric acid dibutylester\n\nThe possible causes behind sudden patchy hair loss, as described in the context, include an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. Other potential causes include tinea capitis, trichotillomania, discoid lupus, and secondary syphilis. However, measures of thyroid-stimulating hormone, vitamin B12, and autoantibodies are only indicated when coexisting disease is suspected.'

#Observations:
1. This response is identical to the one with max_tokens at 1024 or 512.  The response was also more efficient in taking less time.
2. The model likely encounters an EOS sequence flag at this point, so keeping the token size at 256 keeps the model efficient.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,k=10,max_tokens=256)

context list is as follows:
 ['Chapter 324. Traumatic Brain Injury\nIntroduction\nTraumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently\nimpairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily\nCT). Initial treatment consists of ensuring a reliable airway and maintaining adequate\nventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more\nsevere injury to place monitors to track and treat intracranial pressure, decompress the brain if\nintracranial pressure is increased, or remove intracranial hematomas. In the first few days after\nthe injury, maintaining adequate brain perfusion and oxygenation and preventing complications\nof altered sensorium are important. Subsequently, many patients require rehabilitation.\nIn the US, as in much of the world, TBI is a common cause of death and disability. Causes include motor\nvehicle crashes and other transportation-related causes (e

Llama.generate: prefix-match hit


'- Maintaining adequate brain perfusion and oxygenation\n- Preventing complications of altered sensorium\n- Supportive care (preventing systemic complications due to immobilization, providing good nutrition, preventing pressure ulcers)\n- Surgery for patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas\n- Rehabilitation for many patients subsequently\n- No specific treatment, but supportive care should include preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers\n- Drugs are given to prevent seizures if head injury causes significant structural injury or a Glasgow Coma Scale score of < 10\n- Long-term treatment involves multiple drugs for some patients and determining the appropriate dose based on patient tolerance and response\n- Corticosteroids are not recommended for controlling intracranial 

#Observations:
1.  The response covers about 90% of the response for max_token at 512.  Instead of summarizing the rest of the sentence, this llm simply truncates the production of sentences at a certain point mid-way through the sentence.
2.  The creation of the response took lesser time than the response with max tokens at 512.  
3.  On balance, this response could be considered good enough because of its completeness and efficiency of production.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,k=10,max_tokens=256)

context list is as follows:
 ["batting, and layers 2 and 4 are elastic bandages. The injured limb is elevated above the heart for the first\n2 days in a position that allows gravity to help drain edema fluid and thus minimize swelling. After 48 h,\nperiodic application of warmth (eg, a heating pad) for 15 to 20 min may relieve pain and speed healing.\nImmobilization: Immobilization decreases pain and facilitates healing by preventing further injury and is\nhelpful except for very rapidly healing injuries. Joints proximal and distal to the injury should be\nimmobilized.\nA cast is usually used for fractures or other injuries that require weeks of immobilization. Rarely, swelling\nunder a cast is severe enough to contribute to compartment syndrome (see p. 3213). Sometimes, if\nsevere swelling is likely, a cast (and all padding) is cut open from end to end medially and laterally\n(bivalved). Patients with casts should be given written instructions:\n• To keep the cast dry\n• Never to put 

Llama.generate: prefix-match hit


'Based on the context provided, here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:\n\n1. Immobilize the injured limb to prevent further injury and facilitate healing. This can be done using a splint or a cast, depending on the severity of the fracture and the length of immobilization required.\n2. Elevate the injured limb above heart level for the first 48 hours to help drain edema fluid and minimize swelling.\n3. Apply ice to the affected area intermittently during the first 24 to 48 hours to reduce swelling, inflammation, and pain. Use a plastic bag or towel to enclose the ice to prevent direct contact with the skin.\n4. Provide pain relief using analgesics such as acetaminophen or NSAIDs. If pain persists for more than 72 hours after the injury, referral to a specialist is recommended.\n5. Avoid putting any objects inside the cast and keep it dry to prevent infection and skin irritation.\n6. Inspect the edges of the 

#Observations:
1.  This response is decidedly inferior to the previous response, as it provides only the first 6 steps compared to 15 steps provided in the previous response.
2.  This response mainly talks of basic first aid and initial treatment of the fracture. However, it completely misses on after treatment care of the fracture and the recovery.
3.  While this was a more efficient response, it was definitely a less comprehensive response than the previous one.

#Step 5 This time we keep the original chunk size.  We increase the number of top relevant documents retrieved to 10.  We keep the max_tokens to 512 as that response in general created a more comprehensive answer.  We increase the temperature to 0.3 to see if we can get an improved narrative or description of niche conditions in this case.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit ?"
print(generate_rag_response(user_input,k=10,max_tokens=512,temperature=0.3))

context list is as follows:
 ["16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high\nnurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring\nof physiologic parameters.\nSupportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of\ninfection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to\n25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help\ndying patients maintain dignity (see p. 3480).\nPatient Monitoring and Testing\nSome monitoring is manual (ie

Llama.generate: prefix-match hit


Answer:
- Fluid resuscitation with 0.9% normal saline
- Oxygen therapy
- Broad-spectrum antibiotics (modified by culture results)
- Drainage of abscesses and excision of necrotic tissue
- Normalization of blood glucose levels
- Replacement-dose corticosteroids
- Monitoring: ECG, BP, respiratory rate and depth, pulse oximetry, urine flow by indwelling bladder catheter, body temperature, and clinical status.
- Hemodynamic measurements with a central venous or pulmonary artery catheter if the specific type of shock is unclear or when large fluid volumes are needed.
- Early aggressive therapy within 6 hours of suspected diagnosis to improve outcomes.


#Observations:
1.  While this response has same number of bullet points than the earlier response, it introduces some of the measures not seen so far in any of the response to this query.
2.  For example, hemodynamic measures, aggressive therapy, etc are new in this response, while some of the other measures suggested earlier are missing here.
3.  Need domain experts to determine the relevance or accuracy of this response compared to the completely deterministic one with temperature at 0

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine?  If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,k=10,max_tokens=512,temperature=0.3)

context list is as follows:
 ["• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can\nusually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these\ncases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to\nappendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis\nand a normal cecum, the appendix should be removed.\nAppendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For\nnonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics\nshould be continued until the patient's temperature and WBC count have normalized or continued for a\nfixed course, according to the surge

Llama.generate: prefix-match hit


"- Appendicitis symptoms: epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever (rectal temperature 37.7 to 38.3° C [100 to 101° F]) is common.\n- Without surgery or antibiotics, mortality is > 50%. With early surgery, the mortality rate is < 1%, and convalescence is normally rapid and complete. If complications (rupture and develo

#Observations:

1.  This response is more summarized than the previous response for max_tokens at 512.  It answers all the questions well, including whether a surgery should be performed or medicines should be used for treatment.

2.  A domain expert will be needed to validate the suitability of the response for its accuracy.  

3.  On the face of it, the response does look relevant to the query as well as to the context.  However, will need to evaluate this for groundedness and relevance in the subsequent section.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,k=10,max_tokens=512,temperature=0.3)

context list is as follows:
 ['corticosteroids, retinoids, or immunosuppressants.\nHair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be\ndifferent in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium\nis usually temporary as well and abates after the precipitating agent is eliminated.\nKey Points\n• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.\n• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the\nunderlying disorder.\n• Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.\nAlopecia Areata\nAlopecia areata is sudden patchy hair loss in people with no obvious skin or systemic disorder.\nThe scalp and beard are most frequently affected, but any hairy area may be involved. Hair loss may\naffect most or all of the body (alopecia universalis). Alopecia ar

Llama.generate: prefix-match hit


'Based on the context provided, the condition being described is Alopecia Areata. The effective treatments for this condition include:\n- Corticosteroids (triamcinolone acetonide suspension or potent topical corticosteroids)\n- Topical anthralin\n- Minoxidil\n- Induction of allergic contact dermatitis using diphencyprone or squaric acid dibutylester\n\nThe possible causes behind sudden patchy hair loss, as described in the context, include an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. Other potential causes could be tinea capitis, trichotillomania, discoid lupus, and secondary syphilis, but these conditions were not specifically mentioned in relation to sudden patchy hair loss.\n\nTherefore, the answer would be:\n- Effective treatments for sudden patchy hair loss include corticosteroids (triamcinolone acetonide suspension or potent topical corticosteroids), topical anthralin, minoxidil, and induction of allergic contact derma

#Observations:

1.  This response is almost identical to the response with 0 temperature.  
2.  The model is likely operating in the "greedy" mode where the top ranked tokens are so dominant with high confidence and low variance, that a slight increase in temperature has no impact on response creativity.
3.  For this particular case, there is no rationale for increasing the temperature from its completely deterministic value of 0.0

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,k=10,max_tokens=512,temperature=0.3)

context list is as follows:
 ['Chapter 324. Traumatic Brain Injury\nIntroduction\nTraumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently\nimpairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily\nCT). Initial treatment consists of ensuring a reliable airway and maintaining adequate\nventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more\nsevere injury to place monitors to track and treat intracranial pressure, decompress the brain if\nintracranial pressure is increased, or remove intracranial hematomas. In the first few days after\nthe injury, maintaining adequate brain perfusion and oxygenation and preventing complications\nof altered sensorium are important. Subsequently, many patients require rehabilitation.\nIn the US, as in much of the world, TBI is a common cause of death and disability. Causes include motor\nvehicle crashes and other transportation-related causes (e

Llama.generate: prefix-match hit


"- Preventing systemic complications due to immobilization (eg, pneumonia, UTI, thromboembolic disease)\n- Providing good nutrition\n- Preventing pressure ulcers\n- Physical therapy to prevent limb contractures\n- Supportive care for maintaining adequate brain perfusion and oxygenation in the first few days after injury\n- Surgery to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas\n- Rehabilitation services planned early for patients whose coma exceeds 24 hours and have major persistent neurologic sequelae.\n- Prophylactic anticonvulsant considered for patients with significant structural injury (eg, larger contusions or hematomas, brain laceration, depressed skull fracture) or a Glasgow Coma Scale score < 10.\n- Drugs given to prevent seizures if head injury causes significant structural injury or a Glasgow Coma Scale score of < 10. These drugs reduce risk of seizures during the first

#Observations:

1.  In this case as well, this response is almost identical to the response with 0 temperature.  
2.  The model is likely operating in the "greedy" mode where the top ranked tokens are so dominant with high confidence and low variance, that a slight increase in temperature has no impact on response creativity.
3.  For this particular case, there is no rationale for increasing the temperature from its completely deterministic value of 0.0

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,k=10,max_tokens=512,temperature=0.3)

context list is as follows:
 ["batting, and layers 2 and 4 are elastic bandages. The injured limb is elevated above the heart for the first\n2 days in a position that allows gravity to help drain edema fluid and thus minimize swelling. After 48 h,\nperiodic application of warmth (eg, a heating pad) for 15 to 20 min may relieve pain and speed healing.\nImmobilization: Immobilization decreases pain and facilitates healing by preventing further injury and is\nhelpful except for very rapidly healing injuries. Joints proximal and distal to the injury should be\nimmobilized.\nA cast is usually used for fractures or other injuries that require weeks of immobilization. Rarely, swelling\nunder a cast is severe enough to contribute to compartment syndrome (see p. 3213). Sometimes, if\nsevere swelling is likely, a cast (and all padding) is cut open from end to end medially and laterally\n(bivalved). Patients with casts should be given written instructions:\n• To keep the cast dry\n• Never to put 

Llama.generate: prefix-match hit


'Based on the context provided, the following points can be made regarding the care and treatment of a person who has fractured their leg during a hiking trip:\n\n1. Immobilization: The injured limb should be immobilized to prevent further injury and facilitate healing. This can be done using a cast or splint, depending on the severity and type of fracture.\n2. Elevation: The injured limb should be elevated above heart level for the first 48 hours to help drain edema fluid and minimize swelling.\n3. Application of heat: After 48 hours, periodic application of warmth (such as a heating pad) for 15 to 20 minutes may relieve pain and speed healing.\n4. Instructions for cast care: Patients with casts should be given written instructions on how to care for their cast, including keeping it dry, never putting objects inside the cast, inspecting the edges and skin around the cast daily, applying lotion to any red or sore areas, and seeking medical care if an odor emanates from within the cast 

#Observations:

1.  This response is more condensed than the previous response with 512 max allowable tokens.
2.  More importantly, many of the treatment measures here are different compared to what was offered in the previous responses.  While all other responses talked of ice pack, this response talks of warm pad to be put on the affected area.
3.  In general, it has fewer measures for cast and fracture care and recovery.  The response overall seems less comprehensive and has perhaps some creativity / hallunication introduced compared to the previous responses.

#Overall Fine-Tuning related Observations:

1.  Given the balance between comprehensiveness and accuracy of the response, the best combination seems to be the one with source documents, k=10; 512 max_allowed tokens; and temperature at 0.0 to keep it at deterministic response level and not introduce any hallucinations.
2. Context window is kept at 7000 to accommodate the higher number of retreival documents to improve response comprehensiveness.

3.  No need to take the max-allowed tokens to 1024 as in this use case, the model produces almost identically comprehensive response at 512 tokens as well, indicating that it encounters an End of Sequence EOS flag within 512 tokens.

# Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well it has performed in the task.

In [ ]:
groundedness_rater_system_message  = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with the token : ###Question.

The context will begin with the token: ###Context.

The AI generated answer will begin with the token : ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context.

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""


In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with the token : ###Question.

The context will begin with the token : ###Context.

The AI generated answer will begin with the token : ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
#Define a function that combines user queries, the RAG groundedness rater and relevance rater messages
#It then uses them to create prompts for the LLM to rate the model's performance
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message_rag,qna_user_message_rag
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message_rag}\n
                {'user'}: {qna_user_message_rag.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
ground,rel = generate_ground_relevance_response(user_input,k=10,max_tokens=512,temperature=0.0)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key components of the question and context related to managing sepsis in a critical care unit.
2. Check if each component of the AI-generated answer is derived from the information presented in the context.
3. Evaluate the extent to which the metric is followed by the answer.

Steps to explain the answer:
The AI-generated answer includes several components related to managing sepsis in a critical care unit, such as fluid resuscitation, oxygen therapy, broad-spectrum antibiotics, drainage of abscesses and excision of necrotic tissue, normalization of blood glglucose levels, replacement-dose corticosteroids, monitoring, hemodynamic measurements with a central venous or pulmonary artery catheter, adrenal function testing, and early aggressive therapy.

The answer is derived directly from the context as it mentions several times the importance of fluid resuscitation, oxygen therapy, broad-spectrum antibiotics, drainage of abscesses and excisio

#Observations:
1.  Per the evaluation criterion of the llm, the answer has been derived completely from the context.  Hence, the groundedness of the answer is the highest.
2.  Per the evaluation criterion of the llm, the answer directly addresses the question asked.  Hence, the relevance score of the answer is the highest.

3.  The llm does not score the accuracy of the retrieval itself.  It does not judge whether a better search was possible; however, this can be considered high due to the high k value provided to the retreival score.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
ground,rel = generate_ground_relevance_response(user_input,k=10,max_tokens=512,temperature=0.0)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the question parts: What are the common symptoms for appendicitis? Can appendicitis be cured via medicine? If not, what surgical procedure is followed to treat it?
2. Read the context carefully to understand the information related to appendicitis and its treatment.
3. Check if the answer directly addresses each part of the question using only the information provided in the context.
4. Evaluate the extent to which the metric is followed by checking if the answer is derived solely from the context.

The answer adheres to the metric as it lists the common symptoms for appendicitis based on the context and states that appendicitis cannot be cured via medicine and requires surgical removal (appendectomy) according to the information provided in the context. Therefore, the answer is derived only from the context.

Rating: 5 - The metric is followed completely.

 Steps to evaluate the answer:
1. Identify the main aspects of the question. The questi

#Observations:
1.  Once again, per the evaluation criterion of the llm, the answer has been derived completely from the context.  Hence, the groundedness of the answer is the highest.
2.  Per the evaluation criterion of the llm, the answer directly addresses the question asked.  Hence, the relevance score of the answer is the highest.

3.  The llm does not score the accuracy of the retrieval itself.  It does not judge whether a better search was possible; however, this can be considered high due to the high k value provided to the retreival score.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
ground,rel = generate_ground_relevance_response(user_input,k=10,max_tokens=512,temperature=0.0)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the main topic of the question and context. In this case, it is about sudden patchy hair loss and its treatments and possible causes.
2. Check if the AI-generated answer adheres to the metric by deriving information only from the context provided.
3. Examine the answer to ensure that it accurately summarizes the treatments and possible causes mentioned in the context.

The AI-generated answer follows the metric to a good extent as it derives all the information about the treatments for sudden patchy hair loss (alopecia areata) from the context. It also mentions the possible causes, which are discussed in the context. However, the answer could be more specific regarding the causes and their relation to alopecia areata.

Rating: 3 - The metric is followed to a good extent.

 Steps to evaluate the answer:
1. Identify the main aspects of the question. The question asks about effective treatments for sudden patchy hair loss (alopecia areata) and po

#Observations:
1.  Per the judgement of the LLM, the answer does not completely follow the context in presenting the causes and their relation to the scientific disorder for the hair loss patches on the scalp.  It follows the context to a good extent, hence a score of 3 out of 5.
2. Per the judgement of the LLM, the answer addresses the query substantially, and hence receives a perfect score on relevance.
3.  The llm does not score the accuracy of the retrieval itself.  It does not judge whether a better search was possible; however, this can be considered high due to the high k value provided to the retreival score.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
ground,rel = generate_ground_relevance_response(user_input,k=10,max_tokens=512,temperature=0.0)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information in the context related to treatments for traumatic brain injury (TBI).
2. Compare the information in the context with the AI generated answer to determine if the answer is derived only from the context.
3. Evaluate the extent to which the metric is followed based on the comparison.

Explanation:
The context provides detailed information about the initial treatment, supportive care, medications, and rehabilitation for TBI patients. The AI generated answer includes all of these treatments as well as the importance of preventing systemic complications, providing good nutrition, preventing pressure ulcers, and physical therapy to prevent limb contractures.

Comparison:
The AI generated answer matches the information in the context exactly. It includes all of the treatments mentioned in the context and expands on some of them with additional details.

Evaluation:
The metric is followed completely as the answer is derived directl

#Observations:
1.  The LLM rates the response high on groundedness and relevance as it is completely derived from the context and addresses all aspects of the question completely.
2.  Perhaps a different LLM evaluating the response could have rated the answer more critically as we are using the same LLM to create the answer and evaluate its own answer.

3.  The completeness of the manuals in responding to the query, or the completeness of the search from the manuals is not being judged here.  However, the search exhaustiveness could be assumed high due to the high k value used for retrieval. It ensures that the responses even rated slightly less semantically similar also get captured in creating the response.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ground,rel = generate_ground_relevance_response(user_input,k=10,max_tokens=512,temperature=0.0)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to precautions and treatment steps for a person with a fractured leg.
2. Compare each point in the AI generated answer with the corresponding information in the context.
3. Determine if the AI generated answer is derived only from the information presented in the context.

Explanation:
The AI generated answer adheres to the metric as it provides instructions and precautions for a person with a fractured leg based on the information provided in the context. The answer mentions immobilization, keeping the injured area dry, inspecting the cast, applying heat after 48 hours, potential complications of prolonged immobilization, and maintaining good hygiene, all of which are discussed in the context.

Evaluation:
The metric is followed completely as the AI generated answer is derived solely from the information presented in the context.

Rating:
Based on the evaluation criteria, I would rate the answer a 5 as i

#Observations:
1.  The LLM rates the response high on groundedness and relevance as it is completely derived from the context and addresses all aspects of the question completely.
2.  Perhaps a different LLM evaluating the response could have rated the answer more critically as we are using the same LLM to create the answer and evaluate its own answer.

3.  The completeness of the manuals in responding to the query, or the completeness of the search from the manuals is not being judged here.  However, the search exhaustiveness could be assumed high due to the high k value used for retrieval. It ensures that the responses even rated slightly less semantically similar also get captured in creating the response.

#Overall Observations from the Groundedness and relevance score exercise:
1. The fine-tuning resulted in high grounded and relevance scores across the board.
2.  Some of the parameters such as k values for retrival could perhaps be relaxed (lowered) to increase system speed, while not compromising on the response accuracy and comprehensivenss; as well as its groundedness and relevance.

## Actionable Insights and Business Recommendations

1.  A domain expert should be used to rate the relevance of the responses so the system could be fine tuned based on that.  Latest copy of Merck Manuals and more such manuals can be added to the context to improve the thoroughness and richness of the responses.  In general, the context based RAG is expected to perform better than the open internet based AI responses.

2.  A feedback loop should be incorporated to fine tune the model parameters for chunking, retrieval and llm dynamically.
3.  Once fine-tuned, the model can be used in combination with translation transformer models to implement the questions and answers in different languages.

4.  Once fine tuned, this model can be used to assist doctors in high footfall patient systems or where doctor to patient ratios are low. Examples could be rural hospitals, remote health care facilities, public hospitals, or education and charitable hostpitals.  

5.  For efficiency and energy consumption purposes, one can reduce k to a more reasonable number, and can reduce the max token size to 256.  However, the temperature parameter should be kept to zero to ensure accuracy of the responses and remove hallucinations.

6.  The model should be tested with more modern and more efficient models from time to time to check if the new model provides a) better accuracy; b) better efficiency.

7. A more friednly and intuitive user interface should be added to the model to make it easy to query by the busy doctors or their assistants.

<font size=6 color='blue'>Power Ahead</font>
___